In [ ]:
from __future__ import annotations

import copy
import importlib.util
import itertools
import json
import pprint
import re
import resource
import sys
import threading
import time
import warnings
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Iterable

import numpy as np

try:
    import psutil
except ModuleNotFoundError:
    psutil = None


ANALYSIS_INPUT_CELL_MARKER = "# Load the shared configuration file and define the plot requests"


DEFAULT_PAIRED_CUBE_ROOT = Path(
    "/Users/juliomorales/Library/CloudStorage/GoogleDrive-juliomorales1823@gmail.com/"
    "My Drive/Graduate School/Research/Projects/Helioseismology/Vesa_2025/Data"
)

PAIRED_CUBE_SUFFIXES = {".fits", ".fit", ".fts", ".h5", ".hdf5"}


_AGW_FILTER_CACHE: dict[str, Any] = {}


class MemorySampler:
    '''Sample process RSS during batch and notebook analysis work.'''

    def __init__(self, interval_seconds = 0.1):
        self.interval_seconds = float(interval_seconds)
        self.peak_rss = 0
        self._stop = threading.Event()
        self._thread = None
        self._process = psutil.Process() if psutil is not None else None

    def __enter__(self):
        self.start()
        return self

    def __exit__(self, exc_type, exc, tb):
        self.stop()

    def start(self):
        self.peak_rss = self.current_rss()
        self._stop.clear()
        self._thread = threading.Thread(target = self._sample, daemon = True)
        self._thread.start()

    def stop(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join()
        self.peak_rss = max(self.peak_rss, self.current_rss())

    def current_rss(self):
        if self._process is not None:
            return int(self._process.memory_info().rss)
        usage = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        if sys.platform == 'darwin':
            return int(usage)
        return int(usage*1024)

    def _sample(self):
        while not self._stop.is_set():
            self.peak_rss = max(self.peak_rss, self.current_rss())
            time.sleep(self.interval_seconds)


def write_analysis_profile_summary(profile_output_dir, runtime_config, summary):
    '''Persist an optional JSON profile summary for one analysis record.'''

    if profile_output_dir in ['', None]:
        return ''

    output_dir = Path(profile_output_dir).expanduser().resolve()
    output_dir.mkdir(parents = True, exist_ok = True)
    output_name = f"{Path(runtime_config.get('data', {}).get('komega_outfile', 'td_analysis')).stem}_analysis_profile.json"
    output_file = output_dir / output_name
    with output_file.open('w', encoding = 'utf-8') as handle:
        json.dump(summary, handle, indent = 2, sort_keys = True, default = str)

    return str(output_file)


def load_agw_filter_module(config_file: str | Path | None = None) -> Any:
    '''Load the filtering module that owns magnetogram discovery.'''

    module_dir = resolve_td_batch_module_dir(config_file = config_file)
    module_path = module_dir / "filter.py"
    cache_key = str(module_path.resolve())

    if cache_key in _AGW_FILTER_CACHE:
        return _AGW_FILTER_CACHE[cache_key]

    module = load_local_module("time_distance_agw_filter_batch", module_path)
    _AGW_FILTER_CACHE[cache_key] = module

    return module


def is_paired_cube_diagnostic_file(file_path: str | Path) -> bool:
    '''
    Purpose
    -------
    Return whether a file belongs to the observed paired-cube diagnostics.

    Inputs
    ------
    file_path: str | Path
        Candidate file path.

    Outputs
    -------
    is_diagnostic: bool
        Whether the file should be considered for paired-cube batching.
    '''

    path = Path(file_path)
    if path.suffix.lower() not in PAIRED_CUBE_SUFFIXES:
        return False

    name = path.name.lower()
    if "hmimag" in name or "hmicont" in name:
        return False

    if ".int." in name or ".to.hmi.int." in name:
        return False

    if "hmidop" in name:
        return True

    if "aia" in name and "aligned" in name:
        return True

    if ".vel." in name or ".to.hmi.vel." in name:
        return True

    return False


def paired_cube_candidate_priority(file_path: str | Path) -> int | None:
    '''
    Purpose
    -------
    Rank duplicate candidates for the same paired-cube spectral diagnostic.
    '''

    if not is_paired_cube_diagnostic_file(file_path):
        return None

    name = Path(file_path).name.lower()
    priority = 10

    if "aia" in name and "aligned" in name:
        priority = 20
        if ".final." in name or name.endswith(".final.fits"):
            priority = 40

    if ".vel." in name or ".to.hmi.vel." in name:
        priority = 50

    if "hmidop" in name:
        priority = 50

    return priority


def resolve_paired_cube_candidate_key(time_distance_module: Any, file_path: str | Path) -> str:
    '''
    Purpose
    -------
    Resolve one candidate's canonical spectral key using pipeline.py naming logic.
    '''

    spectral_id = time_distance_module.resolve_paired_cube_spectral_identifier(file_path)

    if spectral_id.get("element", "") == "" or spectral_id.get("line", "") == "":
        return ""

    return f"{spectral_id['element'].lower()}{spectral_id['line']}"


def find_directory_magnetogram(directory: str | Path) -> str | None:
    '''
    Purpose
    -------
    Return the unique HMImag FITS file in a directory, if present.
    '''

    return load_agw_filter_module().find_magnetogram(directory, required = False)


def _select_directory_diagnostics(
    directory: Path,
    file_paths: Iterable[Path],
    time_distance_module: Any,
) -> list[dict[str, Any]]:
    '''
    Purpose
    -------
    Select one canonical file per spectral diagnostic within a directory.
    '''

    candidates_by_key: dict[str, list[dict[str, Any]]] = {}

    for file_path in sorted(file_paths):
        priority = paired_cube_candidate_priority(file_path)
        if priority is None:
            continue

        candidate_key = resolve_paired_cube_candidate_key(time_distance_module, file_path)
        if candidate_key == "":
            continue

        candidates_by_key.setdefault(candidate_key, []).append(
            {
                "key": candidate_key,
                "path": str(file_path.resolve()),
                "priority": int(priority),
            }
        )

    selected: list[dict[str, Any]] = []
    for candidate_key in sorted(candidates_by_key):
        candidates = candidates_by_key[candidate_key]
        max_priority = max(candidate["priority"] for candidate in candidates)
        winners = [candidate for candidate in candidates if candidate["priority"] == max_priority]

        if len(winners) > 1:
            raise ValueError(
                "Ambiguous paired_cubes candidates for "
                f"{candidate_key} in {directory}: "
                f"{', '.join(Path(candidate['path']).name for candidate in winners)}"
            )

        selected.append(winners[0])

    selected.sort(key = lambda candidate: (candidate["key"], candidate["path"]))
    return selected


def discover_paired_cube_groups(
    root_path: str | Path,
    time_distance_module: Any | None = None,
) -> list[dict[str, Any]]:
    '''
    Purpose
    -------
    Recursively discover per-directory paired-cube diagnostic groups.
    '''

    if time_distance_module is None:
        time_distance_module = resolve_time_distance_environment()["time_distance_module"]

    root = Path(root_path).expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(f"paired_cubes root does not exist: {root}")
    if not root.is_dir():
        raise NotADirectoryError(f"paired_cubes root is not a directory: {root}")

    files_by_directory: dict[Path, list[Path]] = {}
    for file_path in sorted(path for path in root.rglob("*") if path.is_file()):
        files_by_directory.setdefault(file_path.parent.resolve(), []).append(file_path.resolve())

    groups: list[dict[str, Any]] = []
    for directory in sorted(files_by_directory):
        diagnostics = _select_directory_diagnostics(
            directory,
            files_by_directory[directory],
            time_distance_module,
        )

        if len(diagnostics) == 0:
            continue

        if len(diagnostics) == 1:
            warnings.warn(
                "Skipping paired_cubes directory with only one valid diagnostic cube: "
                f"{directory}",
                RuntimeWarning,
                stacklevel = 2,
            )
            continue

        diagnostic_files = [candidate["path"] for candidate in diagnostics]
        file_pairs = list(itertools.combinations(diagnostic_files, 2))
        groups.append(
            {
                "directory": str(directory),
                "diagnostics": diagnostics,
                "diagnostic_files": diagnostic_files,
                "file_pairs": file_pairs,
                "magnetogram": find_directory_magnetogram(directory),
            }
        )

    return groups


def discover_paired_cubes(
    root_path: str | Path,
    time_distance_module: Any | None = None,
) -> list[tuple[str, str]]:
    '''
    Purpose
    -------
    Recursively discover all deterministic unordered paired-cube file pairs.
    '''

    groups = discover_paired_cube_groups(root_path, time_distance_module = time_distance_module)
    file_pairs: list[tuple[str, str]] = []
    seen: set[tuple[str, str]] = set()

    for group in groups:
        for file_pair in group["file_pairs"]:
            normalized_pair = tuple(file_pair)
            if normalized_pair in seen:
                continue

            seen.add(normalized_pair)
            file_pairs.append(normalized_pair)

    return file_pairs


def _magnetogram_filter_enabled(config: dict[str, Any]) -> bool:
    '''
    Purpose
    -------
    Return whether the runtime config expects magnetogram inputs.
    '''

    filtering = config.get("filtering", {})
    if not bool(filtering.get("enabled", False)):
        return False

    magnetogram = filtering.get("magnetogram", {})
    if not bool(magnetogram.get("enabled", False)):
        return False

    return "magnetogram" in list(filtering.get("filter_sequence", []))


def build_recursive_paired_cubes_batch_plan(
    base_config: dict[str, Any],
    time_distance_module: Any,
    root_path: str | Path,
    *,
    delta_z_km: float | None = None,
    p_dx_Mm: float | None = None,
    dt: float | None = None,
) -> dict[str, Any]:
    '''
    Purpose
    -------
    Build runtime configs for all recursively discovered paired-cube runs.
    '''

    groups = discover_paired_cube_groups(root_path, time_distance_module = time_distance_module)
    magnetogram_required = _magnetogram_filter_enabled(base_config)
    run_configs: list[dict[str, Any]] = []
    file_pairs: list[tuple[str, str]] = []
    skipped_directories: list[dict[str, str]] = []

    for group in groups:
        if magnetogram_required and group["magnetogram"] in ["", None]:
            group["magnetogram"] = load_agw_filter_module().find_magnetogram(group["directory"], required = True)

        for file_pair in group["file_pairs"]:
            run_args = _make_iter_run_args(
                source_type = "paired_cubes",
                file_pair = [file_pair],
                delta_z_km = delta_z_km,
                p_dx_Mm = p_dx_Mm,
                dt = dt,
            )
            run_config = time_distance_module.iter_run_configs(copy.deepcopy(base_config), run_args)[0]
            run_config["data"]["paired_cubes"]["data_dir"] = group["directory"]

            run_configs.append(run_config)
            file_pairs.append(tuple(file_pair))

    if len(run_configs) == 0:
        raise ValueError(f"No valid paired_cubes batch runs were discovered under {Path(root_path).expanduser()}.")

    return {
        "source_type": "paired_cubes",
        "root_path": str(Path(root_path).expanduser().resolve()),
        "directory_records": groups,
        "skipped_directories": skipped_directories,
        "file_pairs": file_pairs,
        "v1_list": [file_1 for file_1, _ in file_pairs],
        "v2_list": [file_2 for _, file_2 in file_pairs],
        "run_configs": run_configs,
    }

_ITER_RUN_ARG_DEFAULTS = {
    "source_type": None,
    "files": None,
    "observable": None,
    "h1": None,
    "h2": None,
    "height_pair": None,
    "file_1": None,
    "file_2": None,
    "file_pair": None,
    "delta_z_km": None,
    "p_dx_Mm": None,
    "dt": None,
}


def load_local_module(module_name: str, file_path: str | Path):
    '''
    Purpose
    -------
    Load local module.
    
    Inputs
    ------
    module_name: str
        Module name to load.
    
    file_path: str | Path
        File path used by this helper.
    
    Outputs
    -------
    loaded_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    file_path = Path(file_path).expanduser().resolve()
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    # Fail early when importlib cannot create a usable module specification.
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load module {module_name!r} from {file_path}.")

    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    # Execute the discovered loader so the imported module is ready for use.
    spec.loader.exec_module(module)
    # Return the imported module object for reuse in the batch helpers.
    return module



def resolve_td_batch_module_dir(config_file: str | Path | None = None) -> Path:
    '''
    Purpose
    -------
    Resolve time-distance batch module dir.
    
    Inputs
    ------
    config_file: str | Path | None
        Path to the configuration file.
    
    Outputs
    -------
    resolved_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Collect candidate directories that might contain the batch notebook and pipeline.
    candidates: list[Path] = []

    # Prefer candidate directories derived from the explicit config file when one was provided.
    if config_file not in ['', None]:
        resolved_config = Path(config_file).expanduser().resolve()
        candidates.extend([
            resolved_config.parent,
            resolved_config.parent / 'Code' / 'Time-Distance',
        ])

    candidates.extend([
        Path.cwd().resolve(),
        (Path.cwd() / 'Code' / 'Time-Distance').resolve(),
    ])

    seen: set[Path] = set()
    # Check each candidate once and return the first valid Time-Distance directory.
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate in seen:
            continue
        seen.add(candidate)

        if (candidate / 'pipeline.py').exists() and (candidate / 'plots.ipynb').exists():
            # Return the assembled helper result.
            return candidate

    raise FileNotFoundError(
        'Could not resolve the Code/Time-Distance directory from the current working directory. '
        'Run batch.ipynb from the project root or from Code/Time-Distance, or set CONFIG_FILE explicitly.'
    )


def resolve_time_distance_environment(config_file: str | Path | None = None) -> dict[str, Any]:
    '''
    Purpose
    -------
    Resolve time distance environment.
    
    Inputs
    ------
    config_file: str | Path | None
        Path to the configuration file.
    
    Outputs
    -------
    resolved_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Resolve the shared batch workspace before deriving any runtime paths.
    module_dir = resolve_td_batch_module_dir(config_file = config_file)
    # Derive the local time-distance script path from the resolved module directory.
    time_distance_file = module_dir / "pipeline.py"
    # Derive the local analysis notebook path from the resolved module directory.
    analysis_notebook = module_dir / "plots.ipynb"
    # Load the local time-distance module directly from the project tree.
    time_distance_module = load_local_module("time_distance_batch_runtime", time_distance_file)
    # Let pipeline.py resolve the active configuration file path.
    resolved_config_file = time_distance_module.resolve_config_file(config_file)

    # Return the assembled helper result.
    return {
        "module_dir": module_dir,
        "time_distance_file": time_distance_file,
        "analysis_notebook": analysis_notebook.resolve(),
        "config_file": resolved_config_file,
        "time_distance_module": time_distance_module,
    }


def load_mode_base_config(
    source_type: str,
    config_file: str | Path | None = None,
) -> tuple[Path, Any, dict[str, Any]]:
    '''
    Purpose
    -------
    Load mode base config.
    
    Inputs
    ------
    source_type: str
        Source-type label used by this helper.
    
    config_file: str | Path | None
        Path to the configuration file.
    
    Outputs
    -------
    loaded_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Load the shared runtime environment and active config for this mode.
    environment = resolve_time_distance_environment(config_file = config_file)
    # Let pipeline.py resolve the active configuration file path.
    resolved_config_file = environment["config_file"]
    time_distance_module = environment["time_distance_module"]
    # Load the Python config module that defines the batch defaults.
    config_module = time_distance_module.load_config_module(resolved_config_file)
    # Normalize the requested source type before selecting its default inputs.
    normalized_source_type = time_distance_module._normalize_source_type(source_type)

    # Prefer the config helper when the module exposes one.
    if hasattr(config_module, "get_config"):
        # Format paired-cube runs using the two file names.
        if normalized_source_type == "paired_cubes":
            # Copy the mode-specific default inputs so the config helper can be called safely.
            default_inputs = copy.deepcopy(getattr(config_module, "paired_cubes_inputs", {}))
        # Format single-cube runs with the observable and sampled heights.
        elif normalized_source_type == "single_cube":
            default_inputs = copy.deepcopy(getattr(config_module, "single_cube_inputs", {}))
        else:
            raise ValueError(f"Unsupported source_type: {source_type}")

        # Keep the editable dispersion-curve settings aligned with the main config file.
        dispersion_curves = copy.deepcopy(getattr(config_module, "dispersion_curve_inputs", None))

        # Build the base config through the config helper so mode defaults are applied consistently.
        base_config = config_module.get_config(
            source_type = normalized_source_type,
            dispersion_curves = dispersion_curves,
            **default_inputs,
        )
    else:
        base_config = copy.deepcopy(config_module.config)
        base_config["data"]["source_type"] = normalized_source_type

    # Return the assembled helper result for downstream batch logic.
    return resolved_config_file, time_distance_module, copy.deepcopy(base_config)

def normalize_batch_source_type(source_type: str) -> str:
    '''Normalize batch source type.'''

    normalized_source_type = str(source_type).strip().lower()
    if normalized_source_type == "single_netcdf_cube":
        return "single_cube"
    return normalized_source_type


def load_config_module_for_batch(config_file: str | Path | None = None) -> tuple[Path, Any, Any]:
    '''Load the active config module for batch-only settings.'''

    environment = resolve_time_distance_environment(config_file = config_file)
    resolved_config_file = environment["config_file"]
    time_distance_module = environment["time_distance_module"]
    config_module = time_distance_module.load_config_module(resolved_config_file)
    return resolved_config_file, time_distance_module, config_module


def load_batch_config(config_file: str | Path | None = None) -> dict[str, Any]:
    '''Load structured batch_config from config.py.'''

    _, _, config_module = load_config_module_for_batch(config_file = config_file)
    return copy.deepcopy(getattr(config_module, "batch_config", {}))


def get_batch_mode_settings(config_file: str | Path | None, source_type: str) -> dict[str, Any]:
    '''Return mode-specific batch settings from config.py.'''

    batch_config = load_batch_config(config_file = config_file)
    normalized_source_type = normalize_batch_source_type(source_type)
    candidate_keys = [str(source_type).strip().lower()]

    if normalized_source_type == "single_cube":
        candidate_keys.extend(["single_cube", "single_netcdf_cube"])
    elif normalized_source_type == "paired_cubes":
        candidate_keys.append("paired_cubes")

    for key in candidate_keys:
        if key in batch_config and isinstance(batch_config[key], dict):
            return copy.deepcopy(batch_config[key])

    return {}


def _as_batch_list(value: Any) -> list[Any]:
    '''Normalize a scalar-or-list batch setting into a list.'''

    if value in ["", None]:
        return []
    if isinstance(value, (str, Path)):
        return [value]
    return list(value)


def get_batch_input_paths(batch_settings: dict[str, Any], *keys: str) -> list[str]:
    '''Collect configured batch input paths from one or more supported keys.'''

    selected_keys = list(keys) if len(keys) > 0 else [
        "input_paths",
        "paths",
        "files",
        "cube_paths",
        "root_paths",
        "directories",
    ]
    input_paths: list[str] = []
    seen: set[str] = set()

    for key in selected_keys:
        for value in _as_batch_list(batch_settings.get(key, [])):
            if value in ["", None]:
                continue
            resolved_path = str(Path(value).expanduser())
            if resolved_path in seen:
                continue
            seen.add(resolved_path)
            input_paths.append(resolved_path)

    return input_paths


def resolve_single_cube_batch_inputs(base_config: dict[str, Any], batch_settings: dict[str, Any]) -> dict[str, Any]:
    '''Resolve single-cube batch inputs from structured batch_config.'''

    single_cube_config = base_config.get("data", {}).get("single_cube", {})
    cube_paths = get_batch_input_paths(batch_settings, "input_paths", "files", "cube_paths")

    if len(cube_paths) == 0 and single_cube_config.get("file", "") not in ["", None]:
        cube_paths = [single_cube_config["file"]]

    observable = batch_settings.get("observable", single_cube_config.get("observable", ""))
    model_atmosphere_path = batch_settings.get(
        "model_atmosphere_path",
        single_cube_config.get("model_atmosphere_path", ""),
    )

    return {
        "cube_paths": cube_paths,
        "observable": observable,
        "model_atmosphere_path": model_atmosphere_path,
    }


def resolve_batch_gaussian_filter_params(base_config: dict[str, Any], batch_settings: dict[str, Any]) -> Any:
    '''Resolve Gaussian filter sweep settings for batch comparisons.'''

    for key in ["gaussian_filter_params", "gaussian_filter_sweep", "gaussian_filters", "filter_params_list"]:
        if key in batch_settings:
            return copy.deepcopy(batch_settings[key])

    return copy.deepcopy(base_config.get("filtering", {}).get("gaussian", {}))


def normalize_paired_cube_pair_record(pair_entry: Any, *, default_delta_z_km: float, default_p_dx_Mm: float, default_dt: float) -> dict[str, Any]:
    '''Normalize one paired-cube batch pair record.'''

    if isinstance(pair_entry, dict):
        metadata = copy.deepcopy(pair_entry)
        if "file_pair" in metadata:
            file_pair = metadata["file_pair"]
        elif "files" in metadata:
            file_pair = metadata["files"]
        else:
            file_pair = [
                metadata.get("v1", metadata.get("file_1", metadata.get("v1_path", ""))),
                metadata.get("v2", metadata.get("file_2", metadata.get("v2_path", ""))),
            ]
    else:
        metadata = {}
        file_pair = pair_entry

    file_pair = list(file_pair)
    if len(file_pair) != 2:
        raise ValueError("Each paired_cubes file-pair entry must contain exactly two paths.")

    v1_path = str(Path(file_pair[0]).expanduser().resolve())
    v2_path = str(Path(file_pair[1]).expanduser().resolve())
    data_dir = metadata.get("data_dir", "")
    if data_dir in ["", None]:
        data_dir = str(Path(v1_path).parent)

    record = {
        "v1": v1_path,
        "v2": v2_path,
        "file_pair": (v1_path, v2_path),
        "data_dir": str(Path(data_dir).expanduser().resolve()),
        "delta_z_km": float(metadata.get("delta_z_km", default_delta_z_km)),
        "p_dx_Mm": float(metadata.get("p_dx_Mm", default_p_dx_Mm)),
        "dt": float(metadata.get("dt", default_dt)),
        "label": str(metadata.get("label", "")),
    }

    return record


def collect_paired_cube_batch_pair_records(batch_settings: dict[str, Any], time_distance_module: Any, *, default_delta_z_km: float, default_p_dx_Mm: float, default_dt: float) -> list[dict[str, Any]]:
    '''Collect paired-cube file pairs from explicit pairs and configured paths.'''

    pair_records: list[dict[str, Any]] = []

    for pair_entry in _as_batch_list(batch_settings.get("file_pairs", [])):
        pair_records.append(
            normalize_paired_cube_pair_record(
                pair_entry,
                default_delta_z_km = default_delta_z_km,
                default_p_dx_Mm = default_p_dx_Mm,
                default_dt = default_dt,
            )
        )

    standalone_files_by_directory: dict[Path, list[Path]] = {}
    for input_path in get_batch_input_paths(batch_settings, "input_paths", "paths", "files", "root_paths", "directories"):
        path = Path(input_path).expanduser().resolve()
        if path.is_dir():
            for group in discover_paired_cube_groups(path, time_distance_module = time_distance_module):
                for file_pair in group["file_pairs"]:
                    pair_records.append(
                        normalize_paired_cube_pair_record(
                            {
                                "file_pair": file_pair,
                                "data_dir": group["directory"],
                            },
                            default_delta_z_km = default_delta_z_km,
                            default_p_dx_Mm = default_p_dx_Mm,
                            default_dt = default_dt,
                        )
                    )
        elif path.is_file():
            standalone_files_by_directory.setdefault(path.parent.resolve(), []).append(path)
        else:
            raise FileNotFoundError(f"Configured paired_cubes input path does not exist: {path}")

    for directory in sorted(standalone_files_by_directory):
        diagnostics = _select_directory_diagnostics(
            directory,
            standalone_files_by_directory[directory],
            time_distance_module,
        )

        if len(diagnostics) == 0:
            continue

        if len(diagnostics) == 1:
            warnings.warn(
                "Skipping paired_cubes directory with only one valid diagnostic cube: "
                f"{directory}",
                RuntimeWarning,
                stacklevel = 2,
            )
            continue

        diagnostic_files = [candidate["path"] for candidate in diagnostics]
        for file_pair in itertools.combinations(diagnostic_files, 2):
            pair_records.append(
                normalize_paired_cube_pair_record(
                    {
                        "file_pair": file_pair,
                        "data_dir": str(directory),
                    },
                    default_delta_z_km = default_delta_z_km,
                    default_p_dx_Mm = default_p_dx_Mm,
                    default_dt = default_dt,
                )
            )

    deduplicated_records: list[dict[str, Any]] = []
    seen_pairs: set[tuple[str, str]] = set()
    for record in pair_records:
        pair_key = tuple(record["file_pair"])
        if pair_key in seen_pairs:
            continue
        seen_pairs.add(pair_key)
        deduplicated_records.append(record)

    if len(deduplicated_records) == 0:
        raise ValueError("No paired_cubes file pairs were found from batch_config.")

    return deduplicated_records


def build_paired_cubes_batch_plan_from_pair_records(base_config: dict[str, Any], time_distance_module: Any, pair_records: Iterable[dict[str, Any]]) -> dict[str, Any]:
    '''Build paired-cube batch runs from normalized pair records.'''

    run_configs: list[dict[str, Any]] = []
    normalized_records = list(pair_records)

    for record in normalized_records:
        run_args = _make_iter_run_args(
            source_type = "paired_cubes",
            file_pair = [record["file_pair"]],
            delta_z_km = record["delta_z_km"],
            p_dx_Mm = record["p_dx_Mm"],
            dt = record["dt"],
        )
        run_config = time_distance_module.iter_run_configs(copy.deepcopy(base_config), run_args)[0]
        run_config["data"]["paired_cubes"]["data_dir"] = record["data_dir"]

        run_configs.append(run_config)

    return {
        "source_type": "paired_cubes",
        "pair_records": normalized_records,
        "file_pairs": [record["file_pair"] for record in normalized_records],
        "v1_list": [record["v1"] for record in normalized_records],
        "v2_list": [record["v2"] for record in normalized_records],
        "run_configs": run_configs,
    }


def build_configured_paired_cubes_batch_plan(base_config: dict[str, Any], time_distance_module: Any, batch_settings: dict[str, Any]) -> dict[str, Any]:
    '''Build paired-cube batch plan directly from config.py batch_config.'''

    paired_defaults = base_config["data"]["paired_cubes"]
    pair_records = collect_paired_cube_batch_pair_records(
        batch_settings,
        time_distance_module,
        default_delta_z_km = float(batch_settings.get("delta_z_km", paired_defaults["delta_z_km"])),
        default_p_dx_Mm = float(batch_settings.get("p_dx_Mm", paired_defaults["p_dx_Mm"])),
        default_dt = float(batch_settings.get("dt", paired_defaults["dt"])),
    )
    plan = build_paired_cubes_batch_plan_from_pair_records(base_config, time_distance_module, pair_records)
    plan["input_paths"] = get_batch_input_paths(batch_settings)
    return plan



def normalize_cube_paths(cube_paths: Iterable[str | Path]) -> list[str]:
    '''
    Purpose
    -------
    Normalize cube paths.
    
    Inputs
    ------
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    Outputs
    -------
    normalized_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Work with normalized paths so duplicate or relative inputs collapse to one entry.
    normalized_paths: list[str] = []
    seen: set[str] = set()

    # Inspect each cube path in turn and collect the derived metadata.
    for cube_path in cube_paths:
        # Ignore empty cube-path placeholders so optional notebook lists remain easy to edit.
        if cube_path in ["", None]:
            continue

        resolved_path = str(Path(cube_path).expanduser().resolve())

        # Skip duplicate paths after normalization so each run is generated only once.
        if resolved_path in seen:
            continue

        seen.add(resolved_path)
        normalized_paths.append(resolved_path)

    # Return the assembled helper result for downstream batch logic.
    return normalized_paths


def generate_unique_unordered_pairs(cube_paths: Iterable[str | Path]) -> list[tuple[str, str]]:
    '''
    Purpose
    -------
    Generate unique unordered pairs.
    
    Inputs
    ------
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    Outputs
    -------
    generated_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Normalize and deduplicate the requested cube paths before building the plan.
    normalized_paths = normalize_cube_paths(cube_paths)
    # Return the assembled helper result for downstream batch logic.
    return [(file_1, file_2) for file_1, file_2 in itertools.combinations(normalized_paths, 2)]


def generate_height_index_pairs(number_of_heights: int) -> list[tuple[int, int]]:
    '''
    Purpose
    -------
    Generate height index pairs.
    
    Inputs
    ------
    number_of_heights: int
        Number of heights used by this helper.
    
    Outputs
    -------
    generated_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    number_of_heights = int(number_of_heights)

    # Reject negative height counts before building combinations.
    if number_of_heights < 0:
        raise ValueError("number_of_heights must be non-negative.")

    # Return the assembled helper result for downstream batch logic.
    return [(h1, h2) for h1, h2 in itertools.combinations(range(number_of_heights), 2)]


def parse_single_cube_field_strength_case(cube_path: str | Path) -> dict[str, Any]:
    '''
    Purpose
    -------
    Parse single cube field-strength case.
    
    Inputs
    ------
    cube_path: str | Path
        Cube path used by this helper.
    
    Outputs
    -------
    parsed_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Resolve the cube path once so the encoded field-strength case can be parsed reliably.
    resolved_path = str(Path(cube_path).expanduser().resolve())
    # Inspect the path components because these comparison helpers infer metadata from folder names.
    path = Path(resolved_path)
    # Initialize the parsed case metadata before scanning the path.
    component = ""
    strength_token = ""

    # Scan the path components to infer the encoded comparison metadata.
    for part in path.parts:
        lower_part = part.lower()

        if lower_part in ["hx", "vx", "z0"]:
            # Initialize the parsed case metadata before scanning the path.
            component = lower_part

        if re.fullmatch(r"\d+(?:[._]\d+)?g", lower_part):
            # Initialize the parsed case metadata before scanning the path.
            strength_token = lower_part

    # Fall back to the encoded path name when the folder scan is not enough.
    if component == "" or strength_token == "":
        match = re.search(
            r"(hx|vx|z0)[_\-/](\d+(?:[._]\d+)?g)",
            resolved_path,
            flags = re.IGNORECASE,
        )

        if match is not None:
            # Initialize the parsed case metadata before scanning the path.
            component = match.group(1).lower()
            strength_token = match.group(2).lower()

    geometry_by_component = {"z0": "zero", "hx": "horizontal", "vx": "vertical"}

    # Limit the field-strength comparison to the supported magnetic geometries.
    if component not in geometry_by_component:
        raise ValueError(
            "Field-strength comparison requires cube paths that encode 'z0', 'hx', or 'vx' geometry. "
            f"Could not infer a supported geometry from {resolved_path}."
        )

    # Require an encoded field strength for this comparison mode.
    if strength_token == "":
        raise ValueError(
            "Field-strength comparison requires cube paths that encode a field-strength folder such as "
            f"'10G'. Could not infer the field strength from {resolved_path}."
        )

    # Convert the encoded field strength into both a numeric value and a stable display label.
    field_strength_G = float(strength_token[:-1].replace("_", "."))
    strength_label_value = (
        f"{int(round(field_strength_G))}"
        if np.isclose(field_strength_G, round(field_strength_G))
        else f"{field_strength_G:g}"
    )

    # Return the assembled helper result for downstream batch logic.
    return {
        "cube_path": resolved_path,
        "component": component,
        "geometry": geometry_by_component[component],
        "field_strength_G": field_strength_G,
        "field_strength_token": strength_token,
        "field_strength_label": f"{strength_label_value} G",
    }


def organize_single_cube_field_strength_cases(cube_paths: Iterable[str | Path]) -> dict[str, Any]:
    '''
    Purpose
    -------
    Organize single cube field-strength cases.
    
    Inputs
    ------
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    Outputs
    -------
    organized_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Normalize and deduplicate the requested cube paths before building the plan.
    normalized_paths = normalize_cube_paths(cube_paths)

    # Require at least one normalized cube path before continuing.
    if len(normalized_paths) == 0:
        raise ValueError("Field-strength comparison requires at least one single_cube path.")

    # Group the parsed field-strength cases by geometry for predictable plotting order.
    cases_by_geometry: dict[str, list[dict[str, Any]]] = {
        "zero": [],
        "horizontal": [],
        "vertical": [],
    }
    # Track geometry-strength combinations so duplicate cases can be rejected early.
    seen_case_keys: dict[tuple[str, str], str] = {}

    # Iterate through the normalized cube paths so every case is handled once.
    for cube_path in normalized_paths:
        case = parse_single_cube_field_strength_case(cube_path)
        case_key = (case["geometry"], case["field_strength_token"])

        # Reject duplicate geometry-strength cases so each comparison panel is unique.
        if case_key in seen_case_keys:
            raise ValueError(
                "Duplicate field-strength comparison case detected for "
                f"{case['geometry']} {case['field_strength_label']}: "
                f"{seen_case_keys[case_key]} and {case['cube_path']}"
            )

        # Track geometry-strength combinations so duplicate cases can be rejected early.
        seen_case_keys[case_key] = case["cube_path"]
        cases_by_geometry[case["geometry"]].append(case)

    # Sort each geometry bucket in place so the plotting order stays stable.
    for geometry_cases in cases_by_geometry.values():
        geometry_cases.sort(key = lambda item: (item["field_strength_G"], item["cube_path"]))

    # Return the assembled helper result for downstream batch logic.
    return {
        "cube_paths": normalized_paths,
        "cases_by_geometry": cases_by_geometry,
        "field_strengths_by_geometry": {
            geometry: [case["field_strength_G"] for case in geometry_cases]
            for geometry, geometry_cases in cases_by_geometry.items()
        },
        "ordered_cases": cases_by_geometry["zero"] + cases_by_geometry["horizontal"] + cases_by_geometry["vertical"],
    }




GAUSSIAN_FILTER_SHAPE_KEYS = ["central_k", "width_k", "central_f", "width_f"]
GAUSSIAN_FILTER_METADATA_KEYS = ["label", "name", "slug"]


def is_gaussian_filter_sequence_value(value: Any) -> bool:
    '''Return whether a Gaussian parameter value is a sweep sequence.'''

    return isinstance(value, (list, tuple, np.ndarray)) and not isinstance(value, (str, bytes))


def expand_gaussian_filter_param_sweep(filter_params: dict[str, Any]) -> list[dict[str, Any]]:
    '''Expand a strict zip-style Gaussian filter parameter sweep.'''

    if not isinstance(filter_params, dict):
        raise TypeError("Each Gaussian filter parameter set must be provided as a dictionary.")

    raw_gaussian = filter_params.get("gaussian", filter_params)
    if not isinstance(raw_gaussian, dict):
        raise TypeError("Gaussian filter parameters must be stored in a dictionary.")

    sequence_shape_keys = [
        key for key in GAUSSIAN_FILTER_SHAPE_KEYS
        if is_gaussian_filter_sequence_value(raw_gaussian.get(key, None))
    ]
    sequence_metadata_keys = [
        key for key in GAUSSIAN_FILTER_METADATA_KEYS
        if is_gaussian_filter_sequence_value(filter_params.get(key, raw_gaussian.get(key, None)))
    ]

    if len(sequence_shape_keys) == 0:
        if len(sequence_metadata_keys) > 0:
            raise ValueError("List-based Gaussian filter labels require list-based shape parameters.")
        return [copy.deepcopy(filter_params)]

    scalar_shape_keys = [key for key in GAUSSIAN_FILTER_SHAPE_KEYS if key not in sequence_shape_keys]
    if len(scalar_shape_keys) > 0:
        raise ValueError(
            "Gaussian filter sweeps must provide central_k, width_k, central_f, and width_f "
            "all as lists of equal length, or all as scalars. "
            f"Scalar/missing keys in a sweep: {', '.join(scalar_shape_keys)}."
        )

    lengths = {key: len(raw_gaussian[key]) for key in GAUSSIAN_FILTER_SHAPE_KEYS}
    if len(set(lengths.values())) != 1:
        raise ValueError(
            "Gaussian filter sweep lists must have equal length. "
            f"Received lengths: {lengths}."
        )

    sweep_length = next(iter(lengths.values()))
    if sweep_length == 0:
        raise ValueError("Gaussian filter sweep lists must contain at least one filter.")

    for key in sequence_metadata_keys:
        metadata_value = filter_params.get(key, raw_gaussian.get(key, None))
        if len(metadata_value) != sweep_length:
            raise ValueError(f"Gaussian filter metadata list '{key}' must have length {sweep_length}.")

    expanded_specs: list[dict[str, Any]] = []
    for filter_index in range(sweep_length):
        expanded_spec = copy.deepcopy(filter_params)
        expanded_gaussian = copy.deepcopy(raw_gaussian)

        for key in GAUSSIAN_FILTER_SHAPE_KEYS:
            expanded_gaussian[key] = raw_gaussian[key][filter_index]

        for key in sequence_metadata_keys:
            metadata_value = filter_params.get(key, raw_gaussian.get(key, None))
            expanded_spec[key] = metadata_value[filter_index]
            expanded_gaussian.pop(key, None)

        if "gaussian" in expanded_spec:
            expanded_spec["gaussian"] = expanded_gaussian
        else:
            expanded_spec = expanded_gaussian
            for key in sequence_metadata_keys:
                metadata_value = filter_params.get(key, raw_gaussian.get(key, None))
                expanded_spec[key] = metadata_value[filter_index]

        expanded_specs.append(expanded_spec)

    return expanded_specs
def normalize_gaussian_filter_param_set(
    filter_params: dict[str, Any],
    *,
    reference_config: dict[str, Any] | None = None,
    filter_index: int | None = None,
) -> dict[str, Any]:
    '''
    Purpose
    -------
    Normalize Gaussian-filter param set.
    
    Inputs
    ------
    filter_params: dict[str, Any]
        Gaussian-filter parameters used by this helper.
    
    reference_config: dict[str, Any] | None
        Reference config used by this helper.
    
    filter_index: int | None
        Filter index used by this helper.
    
    Outputs
    -------
    normalized_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    if not isinstance(filter_params, dict):
        raise TypeError("Each Gaussian filter parameter set must be provided as a dictionary.")

    raw_gaussian = filter_params.get("gaussian", filter_params)
    if not isinstance(raw_gaussian, dict):
        raise TypeError("Gaussian filter parameters must be stored in a dictionary.")

    # Start from the reference Gaussian defaults when they are available.
    gaussian_defaults = {}
    # Inherit any unspecified Gaussian settings from the reference config when available.
    if reference_config is not None:
        # Start from the reference Gaussian defaults when they are available.
        gaussian_defaults = copy.deepcopy(reference_config.get("filtering", {}).get("gaussian", {}))

    # Accumulate the runtime Gaussian settings without mutating the reference config.
    gaussian_config = copy.deepcopy(gaussian_defaults)
    # Copy each runtime Gaussian parameter except the label-style metadata fields.
    for key, value in raw_gaussian.items():
        # Skip display metadata keys because they do not belong in the runtime filter config.
        if key in ["label", "name", "slug"]:
            continue
        # Accumulate the runtime Gaussian settings without mutating the reference config.
        gaussian_config[key] = value

    # Accumulate the runtime Gaussian settings without mutating the reference config.
    gaussian_config["enabled"] = bool(gaussian_config.get("enabled", True))

    # Require the four shape parameters that define the Gaussian filter.
    required_keys = GAUSSIAN_FILTER_SHAPE_KEYS
    missing_keys = [key for key in required_keys if gaussian_config.get(key, None) in ["", None]]
    if len(missing_keys) > 0:
        raise ValueError(
            "Each Gaussian filter parameter set must define "
            f"{', '.join(missing_keys)}."
        )

    sequence_keys = [key for key in required_keys if is_gaussian_filter_sequence_value(gaussian_config.get(key, None))]
    if len(sequence_keys) > 0:
        raise ValueError(
            "A normalized Gaussian filter parameter set cannot contain list-valued shape parameters. "
            "Pass sweep dictionaries through normalize_gaussian_filter_param_list first."
        )

    # Return the assembled helper result for downstream batch logic.
    return {
        "filter_index": None if filter_index is None else int(filter_index),
        "label": str(filter_params.get("label", filter_params.get("name", ""))),
        "gaussian": gaussian_config,
    }


def normalize_gaussian_filter_param_list(

    filter_params_list: Iterable[dict[str, Any]] | dict[str, Any],

    *,

    reference_config: dict[str, Any] | None = None,

) -> list[dict[str, Any]]:

    '''Normalize Gaussian-filter param list.'''



    # Accept either a list of scalar filter dictionaries or one strict zip-style sweep dictionary.

    if isinstance(filter_params_list, dict):

        raw_param_sets = expand_gaussian_filter_param_sweep(filter_params_list)

    else:

        raw_param_sets = []

        for filter_params in filter_params_list:

            raw_param_sets.extend(expand_gaussian_filter_param_sweep(filter_params))



    # Collect the normalized filter specifications in the requested order.

    normalized_filter_params: list[dict[str, Any]] = []



    # Normalize each requested Gaussian filter in the order it was provided.

    for filter_index, filter_params in enumerate(raw_param_sets):

        normalized_filter_params.append(

            normalize_gaussian_filter_param_set(

                filter_params,

                reference_config = reference_config,

                filter_index = filter_index,

            )

        )



    # Require at least one Gaussian filter specification before building the comparison plan.

    if len(normalized_filter_params) == 0:

        raise ValueError("Gaussian-filter comparison requires at least one Gaussian filter parameter set.")



    # Return the assembled helper result for downstream batch logic.

    return normalized_filter_params


def apply_gaussian_filter_params_to_config(
    base_config: dict[str, Any],
    filter_params: dict[str, Any],
) -> dict[str, Any]:
    '''
    Purpose
    -------
    Apply Gaussian-filter params to config.
    
    Inputs
    ------
    base_config: dict[str, Any]
        Base configuration used to build the requested result.
    
    filter_params: dict[str, Any]
        Gaussian-filter parameters used by this helper.
    
    Outputs
    -------
    updated_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Work on a deep copy so shared base-config state is not mutated in place.
    runtime_config = copy.deepcopy(base_config)
    # Normalize the Gaussian filter settings before injecting them into the runtime config.
    normalized_filter_params = normalize_gaussian_filter_param_set(
        filter_params,
        reference_config = runtime_config,
    )

    # Access the filtering section that will be updated with the Gaussian settings.
    filtering = runtime_config.setdefault("filtering", {})
    filtering["gaussian"] = copy.deepcopy(normalized_filter_params["gaussian"])
    filter_sequence_candidates = list(filtering.get("filter_sequence", []))
    # Append the Gaussian stage when it is not already present in the filter order.
    if "gaussian" not in filter_sequence_candidates:
        filter_sequence_candidates.append("gaussian")
    filter_sequence = []
    for filter_name in filter_sequence_candidates:
        filter_config = filtering.get(filter_name, {})
        if bool(filter_config.get("enabled", False)):
            filter_sequence.append(filter_name)
    filtering["filter_sequence"] = filter_sequence
    filtering["enabled"] = len(filter_sequence) > 0

    # Return the assembled helper result.
    return runtime_config


def parse_single_cube_gaussian_filter_comparison_case(cube_path: str | Path) -> dict[str, Any]:
    '''
    Purpose
    -------
    Parse single cube Gaussian-filter comparison case.
    
    Inputs
    ------
    cube_path: str | Path
        Cube path used by this helper.
    
    Outputs
    -------
    parsed_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Resolve the cube path once so the encoded Gaussian-comparison case can be parsed reliably.
    resolved_path = str(Path(cube_path).expanduser().resolve())
    # Inspect the path components because these comparison helpers infer metadata from folder names.
    path = Path(resolved_path)
    # Initialize the parsed case metadata before scanning the path.
    component = ""
    strength_token = ""

    # Scan the path components to infer the encoded comparison metadata.
    for part in path.parts:
        lower_part = part.lower()

        if lower_part in ["hx", "vx", "z0"]:
            # Initialize the parsed case metadata before scanning the path.
            component = lower_part

        if re.fullmatch(r"\d+(?:[._]\d+)?g", lower_part):
            # Initialize the parsed case metadata before scanning the path.
            strength_token = lower_part

    # Fall back to the encoded path name when the folder scan is not enough.
    if component == "" or strength_token == "":
        match = re.search(
            r"(hx|vx|z0)[_\-/](\d+(?:[._]\d+)?g)",
            resolved_path,
            flags = re.IGNORECASE,
        )

        if match is not None:
            # Initialize the parsed case metadata before scanning the path.
            component = match.group(1).lower()
            strength_token = match.group(2).lower()

    # Treat the 0G case as the non-magnetic reference even when the component folder is omitted.
    if component == "" and strength_token == "0g":
        # Initialize the parsed case metadata before scanning the path.
        component = "z0"

    # Fall back to the encoded path name when the folder scan is not enough.
    if component == "" or strength_token == "":
        raise ValueError(
            "Gaussian-filter comparison requires cube paths that encode a supported geometry folder "
            "(hx, vx, or z0) and a field-strength folder such as '10G'. "
            f"Could not infer those values from {resolved_path}."
        )

    # Convert the encoded field strength into both a numeric value and a stable display label.
    field_strength_G = float(strength_token[:-1].replace("_", "."))
    strength_label_value = (
        f"{int(round(field_strength_G))}"
        if np.isclose(field_strength_G, round(field_strength_G))
        else f"{field_strength_G:g}"
    )

    # Map the non-magnetic reference into the canonical comparison label and ordering key.
    if component == "z0" or np.isclose(field_strength_G, 0.0, rtol = 1.0e-9, atol = 1.0e-9):
        case_key = "0g"
        comparison_label = "0G"
        group_name = "zero"
    elif component == "hx":
        # Restrict the magnetic comparison cases to the supported 10G, 50G, and 100G strengths.
        if not any(np.isclose(field_strength_G, value, rtol = 1.0e-9, atol = 1.0e-9) for value in [10.0, 50.0, 100.0]):
            raise ValueError(
                "Gaussian-filter comparison supports only the horizontal 10G, 50G, and 100G cases. "
                f"Received {resolved_path}."
            )

        case_key = f"h{strength_label_value.lower()}g"
        comparison_label = f"h{strength_label_value}G"
        group_name = "horizontal"
    elif component == "vx":
        # Restrict the magnetic comparison cases to the supported 10G, 50G, and 100G strengths.
        if not any(np.isclose(field_strength_G, value, rtol = 1.0e-9, atol = 1.0e-9) for value in [10.0, 50.0, 100.0]):
            raise ValueError(
                "Gaussian-filter comparison supports only the vertical 10G, 50G, and 100G cases. "
                f"Received {resolved_path}."
            )

        case_key = f"v{strength_label_value.lower()}g"
        comparison_label = f"v{strength_label_value}G"
        group_name = "vertical"
    else:
        raise ValueError(
            "Gaussian-filter comparison requires hx, vx, or z0 simulation paths. "
            f"Could not infer a supported case from {resolved_path}."
        )

    # Return the assembled helper result for downstream batch logic.
    return {
        "cube_path": resolved_path,
        "component": component,
        "group_name": group_name,
        "field_strength_G": field_strength_G,
        "field_strength_token": strength_token,
        "field_strength_label": f"{strength_label_value} G",
        "case_key": case_key,
        "comparison_label": comparison_label,
    }


def organize_single_cube_gaussian_filter_comparison_cases(cube_paths: Iterable[str | Path]) -> dict[str, Any]:
    '''
    Purpose
    -------
    Organize single cube Gaussian-filter comparison cases.
    
    Inputs
    ------
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    Outputs
    -------
    organized_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Normalize and deduplicate the requested cube paths before building the plan.
    normalized_paths = normalize_cube_paths(cube_paths)
    # Require at least one normalized cube path before continuing.
    if len(normalized_paths) == 0:
        raise ValueError("Gaussian-filter comparison requires at least one single_cube path.")

    # Keep the required case order explicit because the comparison layout depends on it.
    required_order = ["0g", "h10g", "h50g", "h100g", "v10g", "v50g", "v100g"]
    # Map each parsed case to its canonical comparison key.
    case_lookup: dict[str, dict[str, Any]] = {}

    # Iterate through the normalized cube paths so every case is handled once.
    for cube_path in normalized_paths:
        case = parse_single_cube_gaussian_filter_comparison_case(cube_path)

        if case["case_key"] in case_lookup:
            raise ValueError(
                "Duplicate Gaussian-filter comparison case detected for "
                f"{case['comparison_label']}: {case_lookup[case['case_key']]['cube_path']} and {case['cube_path']}"
            )

        # Map each parsed case to its canonical comparison key.
        case_lookup[case["case_key"]] = case

    missing_cases = [case_key.upper() for case_key in required_order if case_key not in case_lookup]
    if len(missing_cases) > 0:
        missing_labels = [case_lookup_key.replace("H", "h").replace("V", "v") for case_lookup_key in missing_cases]
        raise ValueError(
            "Gaussian-filter comparison requires the seven simulation cases "
            "0G, h10G, h50G, h100G, v10G, v50G, and v100G. "
            f"Missing cases: {', '.join(missing_labels)}."
        )

    # Materialize the cases in the canonical plotting order expected by the notebook.
    ordered_cases = [case_lookup[case_key] for case_key in required_order]

    # Return the assembled helper result for downstream batch logic.
    return {
        "cube_paths": [case["cube_path"] for case in ordered_cases],
        "case_lookup": case_lookup,
        "ordered_cases": ordered_cases,
        "ordered_labels": [case["comparison_label"] for case in ordered_cases],
    }


def infer_shared_single_cube_height_grid_km(
    time_distance_module: Any,
    cube_paths: Iterable[str | Path],
    *,
    rtol: float = 1.0e-9,
    atol: float = 1.0e-6,
) -> list[float]:
    '''
    Purpose
    -------
    Infer shared single cube height grid km.
    
    Inputs
    ------
    time_distance_module: Any
        Time distance module used by this helper.
    
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    rtol: float
        Rtol used by this helper.
    
    atol: float
        Atol used by this helper.
    
    Outputs
    -------
    inferred_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Normalize and deduplicate the requested cube paths before building the plan.
    normalized_paths = normalize_cube_paths(cube_paths)

    # Require at least one normalized cube path before continuing.
    if len(normalized_paths) == 0:
        raise ValueError("Need at least one single_cube path to infer a shared height grid.")

    # Use the first cube as the reference height grid for all subsequent comparisons.
    reference_path = normalized_paths[0]
    # Read the reference height coordinates from the first normalized cube.
    reference_heights = np.asarray(
        time_distance_module.infer_netcdf_height_coordinates_km(reference_path),
        dtype = np.float64,
    )

    # Iterate through the normalized cube paths so every case is handled once.
    for cube_path in normalized_paths[1:]:
        # Read the comparison cube height grid before checking it against the reference.
        comparison_heights = np.asarray(
            time_distance_module.infer_netcdf_height_coordinates_km(cube_path),
            dtype = np.float64,
        )

        # Require matching height grids, while accepting sub-meter rounding drift for plotting plans.
        grids_match = (
            comparison_heights.shape == reference_heights.shape
            and np.allclose(
                comparison_heights,
                reference_heights,
                rtol = float(rtol),
                atol = float(atol),
                equal_nan = True,
            )
        )

        if not grids_match:
            max_abs = (
                float(np.max(np.abs(comparison_heights - reference_heights)))
                if comparison_heights.shape == reference_heights.shape
                else float('inf')
            )
            allowed_tol = max(float(atol), 1.0e-3)

            if np.isfinite(max_abs) and max_abs <= allowed_tol:
                warnings.warn(
                    f"Height grids differ slightly (max diff={max_abs:g} km); accepting for plotting.",
                    RuntimeWarning,
                    stacklevel = 2,
                )
            else:
                raise ValueError(
                    "All single_cube comparison paths must share the same height coordinate grid. "
                    f"{reference_path} and {cube_path} do not match."
                )

    # Return the assembled helper result for downstream batch logic.
    return [float(value) for value in reference_heights]


def generate_shared_single_cube_height_index_pairs(
    time_distance_module: Any,
    cube_paths: Iterable[str | Path],
    *,
    rtol: float = 1.0e-9,
    atol: float = 1.0e-6,
) -> dict[str, Any]:
    '''
    Purpose
    -------
    Generate shared single cube height index pairs.
    
    Inputs
    ------
    time_distance_module: Any
        Time distance module used by this helper.
    
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    rtol: float
        Rtol used by this helper.
    
    atol: float
        Atol used by this helper.
    
    Outputs
    -------
    generated_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Read the shared height grid before enumerating the valid height pairs.
    height_values_km = infer_shared_single_cube_height_grid_km(
        time_distance_module,
        cube_paths,
        rtol = rtol,
        atol = atol,
    )

    # Return the assembled helper result for downstream batch logic.
    return {
        "height_values_km": height_values_km,
        "height_pairs": generate_height_index_pairs(len(height_values_km)),
    }


def _make_iter_run_args(**overrides: Any) -> SimpleNamespace:
    '''
    Purpose
    -------
    Create iter run args.
    
    Inputs
    ------
    overrides: Any
        Overrides used by this helper.
    
    Outputs
    -------
    created_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Start from the canonical iter_run_configs defaults before applying overrides.
    arguments = copy.deepcopy(_ITER_RUN_ARG_DEFAULTS)
    arguments.update(overrides)
    # Return the assembled helper result.
    return SimpleNamespace(**arguments)


def build_paired_cubes_batch_plan(
    base_config: dict[str, Any],
    time_distance_module: Any,
    cube_paths: Iterable[str | Path],
    *,
    delta_z_km: float | None = None,
    p_dx_Mm: float | None = None,
    dt: float | None = None,
) -> dict[str, Any]:
    '''
    Purpose
    -------
    Build paired cubes batch plan.
    
    Inputs
    ------
    base_config: dict[str, Any]
        Base configuration used to build the requested result.
    
    time_distance_module: Any
        Time distance module used by this helper.
    
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    delta_z_km: float | None
        Delta z km used by this helper.
    
    p_dx_Mm: float | None
        P dx Mm used by this helper.
    
    dt: float | None
        Dt used by this helper.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Normalize and deduplicate the requested cube paths before building the plan.
    normalized_paths = normalize_cube_paths(cube_paths)

    # Require at least two unique paths before building paired-cube runs.
    if len(normalized_paths) < 2:
        raise ValueError("paired_cubes batch mode requires at least two unique cube paths.")

    # Build the unique unordered file pairs that feed the paired-cube batch.
    file_pairs = generate_unique_unordered_pairs(normalized_paths)
    # Build the iter_run_configs argument bundle expected by pipeline.py.
    run_args = _make_iter_run_args(
        source_type = "paired_cubes",
        file_pair = file_pairs,
        delta_z_km = delta_z_km,
        p_dx_Mm = p_dx_Mm,
        dt = dt,
    )
    # Expand the batch plan into the concrete runtime configs that will be executed.
    run_configs = time_distance_module.iter_run_configs(copy.deepcopy(base_config), run_args)

    # Return the assembled batch plan and its supporting metadata.
    return {
        "source_type": "paired_cubes",
        "cube_paths": normalized_paths,
        "file_pairs": file_pairs,
        "v1_list": [file_1 for file_1, _ in file_pairs],
        "v2_list": [file_2 for _, file_2 in file_pairs],
        "run_configs": run_configs,
    }


def build_single_cube_batch_plan(
    base_config: dict[str, Any],
    time_distance_module: Any,
    cube_paths: Iterable[str | Path],
    *,
    observable: str | None = None,
    model_atmosphere_path: str | Path | None = None,
) -> dict[str, Any]:
    '''
    Purpose
    -------
    Build single cube batch plan.
    
    Inputs
    ------
    base_config: dict[str, Any]
        Base configuration used to build the requested result.
    
    time_distance_module: Any
        Time distance module used by this helper.
    
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    observable: str | None
        Observable name used by this helper.
    
    model_atmosphere_path: str | Path | None
        Path to the model-atmosphere file.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Normalize and deduplicate the requested cube paths before building the plan.
    normalized_paths = normalize_cube_paths(cube_paths)

    # Require at least one normalized cube path before continuing.
    if len(normalized_paths) == 0:
        raise ValueError("single_cube batch mode requires at least one cube path.")

    # Initialize the run-plan containers that will be filled below.
    run_configs: list[dict[str, Any]] = []
    height_pairs_by_cube: dict[str, list[tuple[int, int]]] = {}
    height_values_km_by_cube: dict[str, list[float]] = {}
    height_pair_rows: list[dict[str, Any]] = []

    # Resolve the optional model-atmosphere path once so every generated config uses the same file.
    resolved_model_atmosphere = (
        str(Path(model_atmosphere_path).expanduser().resolve())
        if model_atmosphere_path not in ["", None]
        else None
    )

    # Iterate through the normalized cube paths so every case is handled once.
    for cube_path in normalized_paths:
        # Read the available height coordinates for the current cube or shared grid.
        height_values_km = [
            float(value)
            for value in time_distance_module.infer_netcdf_height_coordinates_km(cube_path)
        ]
        # Enumerate every valid h1 < h2 pair for the sampled height grid.
        height_pairs = generate_height_index_pairs(len(height_values_km))
        height_pairs_by_cube[cube_path] = height_pairs
        height_values_km_by_cube[cube_path] = height_values_km

        # Iterate over every valid shared height pair.
        for h1_value, h2_value in height_pairs:
            height_pair_rows.append(
                {
                    "file": cube_path,
                    "observable": observable if observable not in ["", None] else base_config["data"]["single_cube"]["observable"],
                    "h1": int(h1_value),
                    "h2": int(h2_value),
                    "h1_km": float(height_values_km[h1_value]),
                    "h2_km": float(height_values_km[h2_value]),
                }
            )

        # Skip cubes that do not provide any valid h1 < h2 combinations.
        if len(height_pairs) == 0:
            continue

        cube_base_config = copy.deepcopy(base_config)
        cube_base_config["data"]["source_type"] = "single_cube"
        cube_base_config["data"]["single_cube"]["file"] = cube_path

        # Apply the observable override only when the caller provided one.
        if observable not in ["", None]:
            cube_base_config["data"]["single_cube"]["observable"] = str(observable)

        # Apply the optional model-atmosphere path when one was supplied.
        if resolved_model_atmosphere is not None:
            cube_base_config["data"]["single_cube"]["model_atmosphere_path"] = resolved_model_atmosphere

        # Build the iter_run_configs argument bundle expected by pipeline.py.
        run_args = _make_iter_run_args(
            source_type = "single_cube",
            files = [cube_path],
            observable = observable,
            height_pair = height_pairs,
        )
        run_configs.extend(time_distance_module.iter_run_configs(cube_base_config, run_args))

    # Return the assembled batch plan and its supporting metadata.
    return {
        "source_type": "single_cube",
        "cube_paths": normalized_paths,
        "height_pairs_by_cube": height_pairs_by_cube,
        "height_values_km_by_cube": height_values_km_by_cube,
        "height_pair_rows": height_pair_rows,
        "run_configs": run_configs,
    }


def build_single_cube_field_strength_comparison_plan(
    base_config: dict[str, Any],
    time_distance_module: Any,
    cube_paths: Iterable[str | Path],
    *,
    observable: str | None = None,
    model_atmosphere_path: str | Path | None = None,
) -> dict[str, Any]:
    '''
    Purpose
    -------
    Build single cube field-strength comparison plan.
    
    Inputs
    ------
    base_config: dict[str, Any]
        Base configuration used to build the requested result.
    
    time_distance_module: Any
        Time distance module used by this helper.
    
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    observable: str | None
        Observable name used by this helper.
    
    model_atmosphere_path: str | Path | None
        Path to the model-atmosphere file.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Organize the supplied cube paths into the canonical case ordering for this comparison mode.
    organized_cases = organize_single_cube_field_strength_cases(cube_paths)
    # Work with normalized paths so duplicate or relative inputs collapse to one entry.
    normalized_paths = organized_cases["cube_paths"]
    # Infer the shared height grid once so every comparison uses the same height pairs.
    shared_heights = generate_shared_single_cube_height_index_pairs(
        time_distance_module,
        normalized_paths,
    )

    # Require at least one shared height pair before building comparison requests.
    if len(shared_heights["height_pairs"]) == 0:
        raise ValueError(
            "Field-strength comparison requires at least two shared heights so h1 < h2 pairs exist."
        )

    # Resolve the optional model-atmosphere path once so every generated config uses the same file.
    resolved_model_atmosphere = (
        str(Path(model_atmosphere_path).expanduser().resolve())
        if model_atmosphere_path not in ["", None]
        else None
    )

    # Resolve the observable once so every run and plot request stays synchronized.
    default_observable = base_config.get("data", {}).get("single_cube", {}).get("observable", "")
    comparison_observable = str(observable) if observable not in ["", None] else str(default_observable)
    # Use one representative cube path because the plotting helper will receive the full case list separately.
    representative_cube_path = organized_cases["ordered_cases"][0]["cube_path"]
    # Accumulate the direct plot requests that plots.ipynb will execute later.
    comparison_runs: list[dict[str, Any]] = []

    # Iterate over every valid shared height pair.
    for h1_value, h2_value in shared_heights["height_pairs"]:
        # Work on a deep copy so shared base-config state is not mutated in place.
        runtime_config = copy.deepcopy(base_config)
        runtime_config["data"]["source_type"] = "single_cube"
        runtime_config["data"].setdefault("single_cube", {})
        runtime_config["data"]["single_cube"]["file"] = representative_cube_path
        runtime_config["data"]["single_cube"]["observable"] = comparison_observable
        runtime_config["data"]["single_cube"]["h1"] = int(h1_value)
        runtime_config["data"]["single_cube"]["h2"] = int(h2_value)

        # Apply the optional model-atmosphere path when one was supplied.
        if resolved_model_atmosphere is not None:
            runtime_config["data"]["single_cube"]["model_atmosphere_path"] = resolved_model_atmosphere

        # Prepare the runtime config so derived output paths and metadata are already resolved.
        prepared_runtime_config = time_distance_module.prepare_runtime_config(runtime_config)
        comparison_runs.append(
            {
                "height_pair": (int(h1_value), int(h2_value)),
                "h1_km": float(shared_heights["height_values_km"][h1_value]),
                "h2_km": float(shared_heights["height_values_km"][h2_value]),
                "runtime_config": prepared_runtime_config,
                "direct_plot_requests": [
                    {
                        "plot_type": "field_strength_comparison",
                        "cube_paths": normalized_paths,
                        "observable": comparison_observable,
                        "h1": int(h1_value),
                        "h2": int(h2_value),
                        "cross_correlation": {"x_label": "Radius [Mm]"},
                        "show": False,
                    }
                ],
            }
        )

    # Return the assembled batch plan and its supporting metadata.
    return {
        "source_type": "single_cube",
        "cube_paths": normalized_paths,
        "organized_cases": organized_cases,
        "height_values_km": shared_heights["height_values_km"],
        "height_pairs": shared_heights["height_pairs"],
        "comparison_runs": comparison_runs,
    }


def build_single_cube_field_orientation_comparison_plan(
    base_config: dict[str, Any],
    time_distance_module: Any,
    cube_paths: Iterable[str | Path],
    *,
    observable: str | None = None,
    model_atmosphere_path: str | Path | None = None,
) -> dict[str, Any]:
    '''
    Purpose
    -------
    Build single cube field-orientation comparison plan.
    
    Inputs
    ------
    base_config: dict[str, Any]
        Base configuration used to build the requested result.
    
    time_distance_module: Any
        Time distance module used by this helper.
    
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    observable: str | None
        Observable name used by this helper.
    
    model_atmosphere_path: str | Path | None
        Path to the model-atmosphere file.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Reuse the field-strength plan and then retarget its direct plot requests.
    comparison_plan = build_single_cube_field_strength_comparison_plan(
        base_config,
        time_distance_module,
        cube_paths,
        observable = observable,
        model_atmosphere_path = model_atmosphere_path,
    )

    comparison_plan = copy.deepcopy(comparison_plan)

    # Retarget each comparison request to the orientation-plot mode.
    for comparison_run in comparison_plan["comparison_runs"]:
        comparison_run["direct_plot_requests"][0]["plot_type"] = "field_orientation_comparison"

    # Return the assembled batch plan and its supporting metadata.
    return comparison_plan


def build_single_cube_gaussian_filter_comparison_plan(
    base_config: dict[str, Any],
    time_distance_module: Any,
    cube_paths: Iterable[str | Path],
    filter_params_list: Iterable[dict[str, Any]],
    *,
    observable: str | None = None,
    model_atmosphere_path: str | Path | None = None,
) -> dict[str, Any]:
    '''
    Purpose
    -------
    Build single cube Gaussian-filter comparison plan.
    
    Inputs
    ------
    base_config: dict[str, Any]
        Base configuration used to build the requested result.
    
    time_distance_module: Any
        Time distance module used by this helper.
    
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    filter_params_list: Iterable[dict[str, Any]]
        Gaussian-filter parameter sets used by this helper.
    
    observable: str | None
        Observable name used by this helper.
    
    model_atmosphere_path: str | Path | None
        Path to the model-atmosphere file.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Organize the supplied cube paths into the canonical case ordering for this comparison mode.
    organized_cases = organize_single_cube_gaussian_filter_comparison_cases(cube_paths)
    # Work with normalized paths so duplicate or relative inputs collapse to one entry.
    normalized_paths = organized_cases["cube_paths"]
    # Normalize the requested Gaussian filter settings before building any run configs.
    normalized_filter_params = normalize_gaussian_filter_param_list(
        filter_params_list,
        reference_config = base_config,
    )
    # Infer the shared height grid once so every comparison uses the same height pairs.
    shared_heights = generate_shared_single_cube_height_index_pairs(
        time_distance_module,
        normalized_paths,
    )

    # Require at least one shared height pair before building comparison requests.
    if len(shared_heights["height_pairs"]) == 0:
        raise ValueError(
            "Gaussian-filter comparison requires at least two shared heights so h1 < h2 pairs exist."
        )

    # Resolve the optional model-atmosphere path once so every generated config uses the same file.
    resolved_model_atmosphere = (
        str(Path(model_atmosphere_path).expanduser().resolve())
        if model_atmosphere_path not in ["", None]
        else None
    )

    # Resolve the observable once so every run and plot request stays synchronized.
    default_observable = base_config.get("data", {}).get("single_cube", {}).get("observable", "")
    comparison_observable = str(observable) if observable not in ["", None] else str(default_observable)
    # Initialize the run-plan containers that will be filled below.
    run_configs: list[dict[str, Any]] = []
    run_rows: list[dict[str, Any]] = []

    # Step through each normalized Gaussian filter setting.
    for filter_params in normalized_filter_params:
        # Iterate through the ordered simulation cases for the current filter setting.
        for case in organized_cases["ordered_cases"]:
            # Iterate over every valid shared height pair.
            for h1_value, h2_value in shared_heights["height_pairs"]:
                # Work on a deep copy so shared base-config state is not mutated in place.
                runtime_config = copy.deepcopy(base_config)
                runtime_config["data"]["source_type"] = "single_cube"
                runtime_config["data"].setdefault("single_cube", {})
                runtime_config["data"]["single_cube"]["file"] = case["cube_path"]
                runtime_config["data"]["single_cube"]["observable"] = comparison_observable
                runtime_config["data"]["single_cube"]["h1"] = int(h1_value)
                runtime_config["data"]["single_cube"]["h2"] = int(h2_value)

                # Apply the optional model-atmosphere path when one was supplied.
                if resolved_model_atmosphere is not None:
                    runtime_config["data"]["single_cube"]["model_atmosphere_path"] = resolved_model_atmosphere

                # Apply the selected Gaussian filter settings to this runtime config copy.
                runtime_config = apply_gaussian_filter_params_to_config(runtime_config, filter_params)
                run_configs.append(runtime_config)
                run_rows.append(
                    {
                        "case_label": case["comparison_label"],
                        "cube_path": case["cube_path"],
                        "filter_index": int(filter_params["filter_index"]),
                        "h1": int(h1_value),
                        "h2": int(h2_value),
                        "h1_km": float(shared_heights["height_values_km"][h1_value]),
                        "h2_km": float(shared_heights["height_values_km"][h2_value]),
                    }
                )

    # Build one representative runtime config that plots.ipynb can reuse for plot generation.
    representative_config = copy.deepcopy(base_config)
    representative_config["data"]["source_type"] = "single_cube"
    representative_config["data"].setdefault("single_cube", {})
    representative_config["data"]["single_cube"]["file"] = organized_cases["ordered_cases"][0]["cube_path"]
    representative_config["data"]["single_cube"]["observable"] = comparison_observable
    representative_config["data"]["single_cube"]["h1"] = int(shared_heights["height_pairs"][0][0])
    representative_config["data"]["single_cube"]["h2"] = int(shared_heights["height_pairs"][0][1])

    # Apply the optional model-atmosphere path when one was supplied.
    if resolved_model_atmosphere is not None:
        # Carry the optional atmosphere override into the representative config as well.
        representative_config["data"]["single_cube"]["model_atmosphere_path"] = resolved_model_atmosphere

    # Apply the selected Gaussian filter settings to this runtime config copy.
    representative_config = apply_gaussian_filter_params_to_config(representative_config, normalized_filter_params[0])
    # Prepare the representative runtime config once before attaching the direct plot requests.
    representative_runtime_config = time_distance_module.prepare_runtime_config(representative_config)

    # Accumulate the direct plot requests that plots.ipynb will execute later.
    comparison_runs: list[dict[str, Any]] = []
    # Iterate over every valid shared height pair.
    for h1_value, h2_value in shared_heights["height_pairs"]:
        # Work on a deep copy so shared base-config state is not mutated in place.
        runtime_config = copy.deepcopy(representative_config)
        runtime_config["data"]["single_cube"]["h1"] = int(h1_value)
        runtime_config["data"]["single_cube"]["h2"] = int(h2_value)
        # Prepare the runtime config so derived output paths and metadata are already resolved.
        prepared_runtime_config = time_distance_module.prepare_runtime_config(runtime_config)
        comparison_runs.append(
            {
                "height_pair": (int(h1_value), int(h2_value)),
                "h1_km": float(shared_heights["height_values_km"][h1_value]),
                "h2_km": float(shared_heights["height_values_km"][h2_value]),
                "runtime_config": prepared_runtime_config,
                "direct_plot_requests": [
                    {
                        "plot_type": "gaussian_filter_comparison",
                        "cube_paths": normalized_paths,
                        "filter_params_list": copy.deepcopy(normalized_filter_params),
                        "observable": comparison_observable,
                        "h1": int(h1_value),
                        "h2": int(h2_value),
                        "cross_correlation": {"x_label": "Radius [Mm]"},
                        "show": False,
                    }
                ],
            }
        )

    # Return the assembled batch plan and its supporting metadata.
    return {
        "source_type": "single_cube",
        "cube_paths": normalized_paths,
        "organized_cases": organized_cases,
        "filter_params_list": normalized_filter_params,
        "height_values_km": shared_heights["height_values_km"],
        "height_pairs": shared_heights["height_pairs"],
        "run_configs": run_configs,
        "run_rows": run_rows,
        "comparison_runs": comparison_runs,
        "representative_runtime_config": representative_runtime_config,
    }



def build_paired_cubes_gaussian_filter_comparison_plan(

    base_config: dict[str, Any],

    time_distance_module: Any,

    pair_records: Iterable[dict[str, Any]],

    filter_params_list: Iterable[dict[str, Any]] | dict[str, Any],

) -> dict[str, Any]:

    '''Build paired-cubes Gaussian-filter comparison plan.'''



    normalized_pair_records = list(pair_records)

    if len(normalized_pair_records) == 0:

        raise ValueError("paired_cubes Gaussian-filter comparison requires at least one paired-cube file pair.")



    normalized_filter_params = normalize_gaussian_filter_param_list(

        filter_params_list,

        reference_config = base_config,

    )



    run_configs: list[dict[str, Any]] = []

    run_rows: list[dict[str, Any]] = []



    for pair_index, pair_record in enumerate(normalized_pair_records):

        for filter_params in normalized_filter_params:

            runtime_config = copy.deepcopy(base_config)

            runtime_config["data"]["source_type"] = "paired_cubes"

            runtime_config["data"].setdefault("paired_cubes", {})

            runtime_config["data"]["paired_cubes"].update(

                {

                    "data_dir": pair_record["data_dir"],

                    "v1": pair_record["v1"],

                    "v2": pair_record["v2"],

                    "file_1": pair_record["v1"],

                    "file_2": pair_record["v2"],

                    "delta_z_km": float(pair_record["delta_z_km"]),

                    "p_dx_Mm": float(pair_record["p_dx_Mm"]),

                    "dt": float(pair_record["dt"]),

                }

            )



            runtime_config = apply_gaussian_filter_params_to_config(runtime_config, filter_params)

            run_configs.append(runtime_config)

            run_rows.append(

                {

                    "pair_index": int(pair_index),

                    "v1": pair_record["v1"],

                    "v2": pair_record["v2"],

                    "filter_index": int(filter_params["filter_index"]),

                    "delta_z_km": float(pair_record["delta_z_km"]),

                }

            )



    representative_runtime_config = time_distance_module.prepare_runtime_config(copy.deepcopy(run_configs[0]))

    comparison_runs = [

        {

            "runtime_config": representative_runtime_config,

            "pair_count": int(len(normalized_pair_records)),

            "filter_count": int(len(normalized_filter_params)),

            "direct_plot_requests": [

                {

                    "plot_type": "paired_cubes_gaussian_filter_comparison",

                    "paired_cases": copy.deepcopy(normalized_pair_records),

                    "filter_params_list": copy.deepcopy(normalized_filter_params),

                    "cross_correlation": {"x_label": "Radius [Mm]"},

                    "show": False,

                }

            ],

        }

    ]



    return {

        "source_type": "paired_cubes",

        "pair_records": normalized_pair_records,

        "filter_params_list": normalized_filter_params,

        "run_configs": run_configs,

        "run_rows": run_rows,

        "comparison_runs": comparison_runs,

        "representative_runtime_config": representative_runtime_config,

    }

def describe_runtime_config(runtime_config: dict[str, Any]) -> str:
    '''
    Purpose
    -------
    Describe runtime config.
    
    Inputs
    ------
    runtime_config: dict[str, Any]
        Runtime config used by this helper.
    
    Outputs
    -------
    output: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Read the data block once so the remaining formatting logic stays concise.
    data = runtime_config.get("data", {})
    source_type = str(data.get("source_type", "")).strip().lower()
    xcorr_geometry = str(runtime_config.get("time_distance", {}).get("xcorr_geometry", "annulus")).strip().lower()
    geometry_label = "" if xcorr_geometry in ["", "annulus"] else f" | {xcorr_geometry.title()} wedge"

    # Format paired-cube runs using the two file names.
    if source_type == "paired_cubes":
        # Return the concise human-readable label for this runtime config.
        return f"{Path(data.get('v1', '')).name} vs {Path(data.get('v2', '')).name}{geometry_label}"

    # Format single-cube runs with the observable and sampled heights.
    if source_type == "single_cube":
        h1_km = data.get("resolved_h1_km", data.get("h1", "?"))
        h2_km = data.get("resolved_h2_km", data.get("h2", "?"))
        observable = data.get("observable", "")
        # Return the concise human-readable label for this runtime config.
        return f"{Path(data.get('file', '')).name} | {observable} | {h1_km:g} km vs {h2_km:g} km{geometry_label}"

    # Return the concise human-readable label for this runtime config.
    return source_type or "unknown run"


def runtime_output_paths(runtime_config: dict[str, Any]) -> dict[str, Path]:
    '''
    Purpose
    -------
    Runtime output paths.
    
    Inputs
    ------
    runtime_config: dict[str, Any]
        Runtime config used by this helper.
    
    Outputs
    -------
    output: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Read the data block once so the remaining formatting logic stays concise.
    data = runtime_config.get("data", {})
    # Collect the standard output products emitted by pipeline.py for this run.
    output_files = {
        "outfile": data.get("outfile", ""),
        "phase_outfile": data.get("phase_outfile", ""),
        "komega_outfile": data.get("komega_outfile", ""),
        "coherence_outfile": data.get("coherence_outfile", ""),
        "orientation_validation_outfile": data.get("orientation_validation_outfile", ""),
    }

    # Return the resolved output paths that belong to this runtime config.
    return {
        key: Path(value).expanduser().resolve()
        for key, value in output_files.items()
        if value not in ["", None]
    }


def batch_outputs_exist(runtime_config: dict[str, Any]) -> bool:
    '''
    Purpose
    -------
    Batch outputs exist.
    
    Inputs
    ------
    runtime_config: dict[str, Any]
        Runtime config used by this helper.
    
    Outputs
    -------
    output: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Resolve the declared output files before deciding whether the run can be reused.
    output_paths = runtime_output_paths(runtime_config)

    # A run with no declared outputs cannot be considered reusable.
    if len(output_paths) == 0:
        # Return the assembled helper result.
        return False

    # Return the assembled helper result.
    return all(path.exists() for path in output_paths.values())


def execute_batch_runs(
    time_distance_module: Any,
    config_file: str | Path,
    run_configs: Iterable[dict[str, Any]],
    *,
    skip_existing: bool = True,
    continue_on_error: bool = False,
    verbose: bool = True,
) -> list[dict[str, Any]]:
    '''
    Purpose
    -------
    Execute batch runs.
    
    Inputs
    ------
    time_distance_module: Any
        Time distance module used by this helper.
    
    config_file: str | Path
        Path to the configuration file.
    
    run_configs: Iterable[dict[str, Any]]
        Run configurations used by this helper.
    
    skip_existing: bool
        Whether existing outputs should be reused.
    
    continue_on_error: bool
        Whether processing should continue after an error.
    
    verbose: bool
        Verbose used by this helper.
    
    Outputs
    -------
    execution_result: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Expand the batch plan into the concrete runtime configs that will be executed.
    run_configs = list(run_configs)
    # Materialize the run list so progress counts and later summaries stay stable.
    run_records: list[dict[str, Any]] = []
    # Record the total number of runs once for progress reporting and result metadata.
    total_runs = len(run_configs)

    # Execute each planned runtime config and collect its batch record.
    for run_index, run_config in enumerate(run_configs, start = 1):
        # Work on a deep copy so shared base-config state is not mutated in place.
        runtime_config = time_distance_module.prepare_runtime_config(copy.deepcopy(run_config))
        # Resolve the configured output files before checking whether the run is reusable.
        output_paths = {
            key: str(path)
            for key, path in runtime_output_paths(runtime_config).items()
        }
        # Initialize the batch record that will capture status, labels, and output paths.
        record = {
            "run_index": int(run_index),
            "total_runs": int(total_runs),
            "status": "",
            "label": describe_runtime_config(runtime_config),
            "source_type": runtime_config["data"]["source_type"],
            "runtime_config": runtime_config,
            "output_paths": output_paths,
        }

        # Print progress only when verbose output is enabled.
        if verbose:
            # Emit a concise progress update for the current batch stage.
            print(f"[{run_index}/{total_runs}] {record['label']}")

        # Reuse completed outputs when the batch is allowed to skip existing files.
        if skip_existing and batch_outputs_exist(runtime_config):
            record["status"] = "skipped_existing"

            # Print progress only when verbose output is enabled.
            if verbose:
                # Emit a concise progress update for the current batch stage.
                print("  skipped existing outputs")

            run_records.append(record)
            continue

        # Run the time-distance pipeline and convert any failure into an explicit batch record.
        try:
            results = time_distance_module.run_time_distance(
                config_file = config_file,
                config_override = copy.deepcopy(run_config),
            )
            record["status"] = "completed"
            record["result_shapes"] = {
                "xc": tuple(results["xc"].shape),
                "phase_diff": tuple(results["phase_diff"].shape),
                "komega": tuple(results["komega"].shape),
                "coherence": tuple(results["coherence"].shape),
            }

            # Print progress only when verbose output is enabled.
            if verbose:
                # Emit a concise progress update for the current batch stage.
                print(f"  wrote {record['output_paths']['outfile']}")
        except Exception as exc:
            record["status"] = "failed"
            record["error"] = f"{exc.__class__.__name__}: {exc}"

            # Print progress only when verbose output is enabled.
            if verbose:
                # Emit a concise progress update for the current batch stage.
                print(f"  failed: {record['error']}")

            run_records.append(record)

            if not continue_on_error:
                raise

            continue

        run_records.append(record)

    # Return the collected execution records for downstream notebook cells.
    return run_records


def build_td_analysis_input_cell_source(
    runtime_config: dict[str, Any],
    *,
    use_config: bool = True,
    direct_plot_requests: list[dict[str, Any]] | list[str] | None = None,
) -> str:
    '''
    Purpose
    -------
    Build time-distance analysis input cell source.
    
    Inputs
    ------
    runtime_config: dict[str, Any]
        Runtime config used by this helper.
    
    use_config: bool
        Whether config-driven requests should be used.
    
    direct_plot_requests: list[dict[str, Any]] | list[str] | None
        Direct plot requests supplied by the caller.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Serialize the runtime config and direct requests into readable notebook literals.
    runtime_literal = pprint.pformat(copy.deepcopy(runtime_config), sort_dicts = False, width = 100)
    plot_request_literal = pprint.pformat(
        copy.deepcopy([] if direct_plot_requests is None else direct_plot_requests),
        sort_dicts = False,
        width = 100,
    )

    # Return the notebook cell source that injects the runtime config into plots.ipynb.
    return (
        "# Load the shared configuration file and define the plot requests\n"
        "config_file = resolve_config_file()\n"
        "modules = load_project_modules(config_file)\n"
        f"runtime_config = {runtime_literal}\n"
        "config = prepare_runtime_config(runtime_config)\n\n"
        "# Use a wider k-omega range for simulation cubes\n"
        "if config.get('data', {}).get('source_type', '') == 'single_cube':\n"
        "    PLOT_DEFAULTS['komega_diagram']['vmin'] = -80.0\n\n"
        "# Comparison-plot execution defaults\n"
        "comparison_execution_mode = 'load'\n"
        "comparison_missing_data_behavior = 'error'\n\n"
        "config['profile_enabled'] = bool(runtime_config.get('profile_enabled', False))\n"
        "config['profile_output_dir'] = runtime_config.get('profile_output_dir', '')\n"
        "config['show_plots'] = bool(runtime_config.get('show_plots', False))\n"
        "config['close_figures_after_save'] = bool(runtime_config.get('close_figures_after_save', True))\n\n"
        f"use_config = {bool(use_config)}\n"
        f"direct_plot_requests = {plot_request_literal}\n\n"
        "print(config_file)\n"
        "print(config['data']['outfile'])\n"
    )


def import_td_analysis_notebook_dependencies():
    '''
    Purpose
    -------
    Import time-distance analysis notebook dependencies.
    
    Inputs
    ------
    None
    
    Outputs
    -------
    output: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Import the notebook execution packages lazily so the rest of the batch helpers stay usable.
    try:
        import nbformat
        from nbclient import NotebookClient
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Executing plots.ipynb requires the 'nbformat' and 'nbclient' packages."
        ) from exc

    # Return the imported notebook-execution helpers.
    return nbformat, NotebookClient


def execute_td_analysis_notebook(
    runtime_config: dict[str, Any],
    *,
    analysis_notebook: str | Path | None = None,
    output_dir: str | Path | None = None,
    use_config: bool = True,
    direct_plot_requests: list[dict[str, Any]] | list[str] | None = None,
    timeout: int = 3600,
    kernel_name: str | None = None,
) -> Path:
    '''
    Purpose
    -------
    Execute time-distance analysis notebook.
    
    Inputs
    ------
    runtime_config: dict[str, Any]
        Runtime config used by this helper.
    
    analysis_notebook: str | Path | None
        Path to the analysis notebook that will be executed.
    
    output_dir: str | Path | None
        Output directory used by this helper.
    
    use_config: bool
        Whether config-driven requests should be used.
    
    direct_plot_requests: list[dict[str, Any]] | list[str] | None
        Direct plot requests supplied by the caller.
    
    timeout: int
        Execution timeout in seconds.
    
    kernel_name: str | None
        Kernel name used by this helper.
    
    Outputs
    -------
    execution_result: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Load the notebook execution stack only when analysis execution is requested.
    nbformat, NotebookClient = import_td_analysis_notebook_dependencies()

    if analysis_notebook in ["", None]:
        # Resolve the shared batch workspace before deriving any runtime paths.
        analysis_notebook_path = (resolve_td_batch_module_dir() / "plots.ipynb").resolve()
    else:
        # Resolve the source analysis notebook path, defaulting to the local project copy.
        analysis_notebook_path = Path(analysis_notebook).expanduser().resolve()

    # Load the source analysis notebook from disk before rewriting its input cell.
    with analysis_notebook_path.open("r", encoding = "utf-8") as notebook_handle:
        notebook = nbformat.read(notebook_handle, as_version = 4)

    target_cell_index = None
    # Find the input-cell marker so the runtime config can be injected into the notebook.
    for index, cell in enumerate(notebook.cells):
        if cell.get("cell_type") == "code" and ANALYSIS_INPUT_CELL_MARKER in cell.get("source", ""):
            target_cell_index = index
            break

    # Fail loudly when the marker cell cannot be found in the analysis notebook.
    if target_cell_index is None:
        raise ValueError(
            f"Could not find the td_analysis input cell marker in {analysis_notebook_path}."
        )

    # Inject the runtime-specific input cell so the executed notebook uses this batch config.
    notebook.cells[target_cell_index]["source"] = build_td_analysis_input_cell_source(
        runtime_config,
        use_config = use_config,
        direct_plot_requests = direct_plot_requests,
    )

    # Default executed notebooks to an analysis_notebooks directory beside the data outputs.
    if output_dir in ["", None]:
        output_directory = (
            Path(runtime_config["paths"]["data_output_dir"]).expanduser().resolve()
            / "analysis_notebooks"
        )
    else:
        output_directory = Path(output_dir).expanduser().resolve()

    output_directory.mkdir(parents = True, exist_ok = True)
    # Build the output notebook path from the runtime config so each analysis run is traceable.
    executed_notebook = output_directory / f"{Path(runtime_config['data']['komega_outfile']).stem}_plots.ipynb"

    # Configure the notebook client to run from the analysis notebook directory.
    client_kwargs = {
        "timeout": int(timeout),
        "resources": {"metadata": {"path": str(analysis_notebook_path.parent)}},
    }

    # Honor an explicit kernel override when one was supplied.
    if kernel_name not in ["", None]:
        # Add the explicit kernel name only when the caller requested one.
        client_kwargs["kernel_name"] = str(kernel_name)

    # Create the notebook client that will execute the modified analysis notebook.
    client = NotebookClient(notebook, **client_kwargs)
    client.execute()

    # Persist the executed notebook to disk after the client finishes running it.
    with executed_notebook.open("w", encoding = "utf-8") as notebook_handle:
        nbformat.write(notebook, notebook_handle)

    # Return the path to the executed analysis notebook.
    return executed_notebook


def load_td_analysis_in_process_runtime(analysis_notebook: str | Path | None = None):
    '''Load td_analysis helper cells once for in-process batch plotting.'''

    nbformat_module, _ = import_td_analysis_notebook_dependencies()
    if analysis_notebook in ['', None]:
        analysis_notebook_path = (resolve_td_batch_module_dir() / 'plots.ipynb').resolve()
    else:
        analysis_notebook_path = Path(analysis_notebook).expanduser().resolve()

    with analysis_notebook_path.open('r', encoding = 'utf-8') as notebook_handle:
        notebook = nbformat_module.read(notebook_handle, as_version = 4)

    helper_sources = []
    for cell in notebook.cells:
        if cell.get('cell_type') != 'code':
            continue
        source = cell.get('source', '')
        if 'def parse_spectral_identifier(' in source and 'def prepare_runtime_config(' in source:
            helper_sources.append(source)
        elif 'def parse_single_cube_field_strength_case(' in source and 'def build_field_strength_comparison_column_cases(' in source:
            helper_sources.append(source)

    if len(helper_sources) < 2:
        raise RuntimeError(f'Could not find the td_analysis helper cells in {analysis_notebook_path}.')

    module = SimpleNamespace()
    namespace = module.__dict__
    for source in helper_sources:
        exec(compile(source, str(analysis_notebook_path), 'exec'), namespace)

    namespace['_PLOT_DEFAULTS_TEMPLATE'] = copy.deepcopy(namespace.get('PLOT_DEFAULTS', {}))
    namespace['_analysis_notebook_path'] = analysis_notebook_path
    return module


def execute_td_analysis_in_process(
    analysis_runtime,
    runtime_config: dict[str, Any],
    *,
    config_file: str | Path | None = None,
    use_config: bool = True,
    direct_plot_requests: list[dict[str, Any]] | list[str] | None = None,
) -> dict[str, Any]:
    '''Run td_analysis plotting helpers in the current Python process.'''

    namespace = analysis_runtime.__dict__
    namespace['PLOT_DEFAULTS'] = copy.deepcopy(namespace.get('_PLOT_DEFAULTS_TEMPLATE', {}))

    component_times = {}
    plot_times = {}
    plot_call_counts = {}
    generated_products = []

    setup_start = time.perf_counter()
    if config_file in ['', None]:
        resolved_config_file = resolve_td_batch_module_dir() / 'config.py'
    else:
        resolved_config_file = Path(config_file).expanduser().resolve()
    modules = namespace['load_project_modules'](resolved_config_file)
    config = namespace['prepare_runtime_config'](copy.deepcopy(runtime_config))
    config['profile_enabled'] = bool(runtime_config.get('profile_enabled', False))
    config['profile_output_dir'] = runtime_config.get('profile_output_dir', '')
    config['show_plots'] = bool(runtime_config.get('show_plots', False))
    config['close_figures_after_save'] = bool(runtime_config.get('close_figures_after_save', True))
    if config.get('data', {}).get('source_type', '') == 'single_cube':
        namespace['PLOT_DEFAULTS']['komega_diagram']['vmin'] = -80.0
    component_times['setup'] = time.perf_counter() - setup_start

    request_start = time.perf_counter()
    plot_requests = namespace['build_plot_requests'](
        config,
        use_config = use_config,
        direct_plot_requests = direct_plot_requests,
        comparison_execution_mode = 'load',
        comparison_missing_data_behavior = 'error',
    )
    component_times['build_plot_requests'] = time.perf_counter() - request_start

    plot_cache = {}
    for request in plot_requests:
        plot_type = request['plot_type']
        plot_start = time.perf_counter()
        result = namespace['run_plot_request'](request, config, modules, plot_cache)
        plot_elapsed = time.perf_counter() - plot_start
        plot_times[plot_type] = plot_times.get(plot_type, 0.0) + plot_elapsed
        plot_call_counts[plot_type] = plot_call_counts.get(plot_type, 0) + 1

        product = {'plot_type': plot_type}
        if isinstance(result, dict):
            for key in ['saved_file', 'data_file']:
                if result.get(key, None) is not None:
                    product[key] = str(result[key])
        else:
            product['result'] = str(result)
        generated_products.append(product)

    return {
        'generated_products': generated_products,
        'component_times_seconds': component_times,
        'plot_times_seconds': plot_times,
        'plot_call_counts': plot_call_counts,
    }


def execute_td_analysis_for_batch(
    run_records: Iterable[dict[str, Any]],
    *,
    analysis_notebook: str | Path | None = None,
    output_dir: str | Path | None = None,
    use_config: bool = True,
    direct_plot_requests: list[dict[str, Any]] | list[str] | None = None,
    timeout: int = 3600,
    kernel_name: str | None = None,
    continue_on_error: bool = False,
    verbose: bool = True,
    analysis_backend: str = 'notebook',
    profile_enabled: bool = False,
    profile_output_dir: str | Path | None = None,
    config_file: str | Path | None = None,
) -> list[dict[str, Any]]:
    '''
    Purpose
    -------
    Execute time-distance analysis for batch.
    '''

    analysis_backend = str(analysis_backend).strip().lower()
    if analysis_backend not in ['notebook', 'in_process']:
        raise ValueError("analysis_backend must be either 'notebook' or 'in_process'.")

    run_records = list(run_records)
    analysis_records: list[dict[str, Any]] = []
    runnable_records = [
        record for record in run_records if record.get('status', '') in ['completed', 'skipped_existing']
    ]
    missing_dependency_error = None
    analysis_runtime = None

    try:
        import_td_analysis_notebook_dependencies()
        if analysis_backend == 'in_process':
            analysis_runtime = load_td_analysis_in_process_runtime(analysis_notebook)
    except ModuleNotFoundError as exc:
        missing_dependency_error = f'{exc.__class__.__name__}: {exc}'

    if missing_dependency_error is not None:
        if verbose:
            print(f'Analysis notebook execution unavailable: {missing_dependency_error}')

        for analysis_index, record in enumerate(runnable_records, start = 1):
            analysis_records.append({
                'analysis_index': int(analysis_index),
                'total_runs': int(len(runnable_records)),
                'label': record['label'],
                'source_type': record['source_type'],
                'runtime_config': record['runtime_config'],
                'status': 'skipped_missing_dependency',
                'error': missing_dependency_error,
                'analysis_backend': analysis_backend,
                'wall_seconds': 0.0,
                'peak_rss_bytes': 0,
                'component_times_seconds': {},
                'plot_times_seconds': {},
                'profile_summary_file': '',
            })

        for record in run_records:
            if record.get('status', '') == 'failed':
                analysis_records.append({
                    'analysis_index': None,
                    'total_runs': int(len(runnable_records)),
                    'label': record['label'],
                    'source_type': record['source_type'],
                    'runtime_config': record['runtime_config'],
                    'status': 'skipped_parent_failure',
                    'analysis_backend': analysis_backend,
                    'wall_seconds': 0.0,
                    'peak_rss_bytes': 0,
                    'component_times_seconds': {},
                    'plot_times_seconds': {},
                    'profile_summary_file': '',
                })
        return analysis_records

    for analysis_index, record in enumerate(runnable_records, start = 1):
        runtime_config = copy.deepcopy(record['runtime_config'])
        if analysis_backend == 'in_process':
            runtime_config.setdefault('show_plots', False)
            runtime_config.setdefault('close_figures_after_save', True)
        if profile_enabled:
            runtime_config['profile_enabled'] = True
            if profile_output_dir not in ['', None]:
                runtime_config['profile_output_dir'] = str(Path(profile_output_dir).expanduser().resolve())

        analysis_record = {
            'analysis_index': int(analysis_index),
            'total_runs': int(len(runnable_records)),
            'label': record['label'],
            'source_type': record['source_type'],
            'runtime_config': runtime_config,
            'status': '',
            'analysis_backend': analysis_backend,
            'wall_seconds': 0.0,
            'peak_rss_bytes': 0,
            'component_times_seconds': {},
            'plot_times_seconds': {},
            'profile_summary_file': '',
        }

        if verbose:
            print(f"[analysis {analysis_index}/{len(runnable_records)}] {record['label']}")

        try:
            with MemorySampler() as memory_sampler:
                start_time = time.perf_counter()
                if analysis_backend == 'notebook':
                    executed_notebook = execute_td_analysis_notebook(
                        runtime_config,
                        analysis_notebook = analysis_notebook,
                        output_dir = output_dir,
                        use_config = use_config,
                        direct_plot_requests = direct_plot_requests,
                        timeout = timeout,
                        kernel_name = kernel_name,
                    )
                    elapsed = time.perf_counter() - start_time
                    analysis_record['executed_notebook'] = str(executed_notebook)
                    analysis_record['component_times_seconds'] = {'notebook_execute': float(elapsed)}
                else:
                    in_process_result = execute_td_analysis_in_process(
                        analysis_runtime,
                        runtime_config,
                        config_file = config_file,
                        use_config = use_config,
                        direct_plot_requests = direct_plot_requests,
                    )
                    elapsed = time.perf_counter() - start_time
                    analysis_record['generated_products'] = in_process_result['generated_products']
                    analysis_record['component_times_seconds'] = in_process_result['component_times_seconds']
                    analysis_record['plot_times_seconds'] = in_process_result['plot_times_seconds']
                    analysis_record['plot_call_counts'] = in_process_result['plot_call_counts']

            analysis_record['status'] = 'completed'
            analysis_record['wall_seconds'] = float(elapsed)
            analysis_record['peak_rss_bytes'] = int(memory_sampler.peak_rss)
            if profile_enabled:
                analysis_record['profile_summary_file'] = write_analysis_profile_summary(
                    profile_output_dir,
                    runtime_config,
                    analysis_record,
                )

            if verbose:
                if analysis_backend == 'notebook':
                    print(f"  executed {analysis_record['executed_notebook']}")
                else:
                    print(f"  generated {len(analysis_record.get('generated_products', []))} plot product(s)")
        except Exception as exc:
            analysis_record['status'] = 'failed'
            analysis_record['error'] = f'{exc.__class__.__name__}: {exc}'
            analysis_records.append(analysis_record)
            if verbose:
                print(f"  failed: {analysis_record['error']}")
            if not continue_on_error:
                raise
            continue

        analysis_records.append(analysis_record)

    for record in run_records:
        if record.get('status', '') == 'failed':
            analysis_records.append({
                'analysis_index': None,
                'total_runs': int(len(runnable_records)),
                'label': record['label'],
                'source_type': record['source_type'],
                'runtime_config': record['runtime_config'],
                'status': 'skipped_parent_failure',
                'analysis_backend': analysis_backend,
                'wall_seconds': 0.0,
                'peak_rss_bytes': 0,
                'component_times_seconds': {},
                'plot_times_seconds': {},
                'profile_summary_file': '',
            })

    return analysis_records


def summarize_batch_records(
    run_records: Iterable[dict[str, Any]],
    analysis_records: Iterable[dict[str, Any]] | None = None,
) -> dict[str, dict[str, int]]:
    '''
    Purpose
    -------
    Summarize batch records.
    
    Inputs
    ------
    run_records: Iterable[dict[str, Any]]
        Run records used by this helper.
    
    analysis_records: Iterable[dict[str, Any]] | None
        Analysis records used by this helper.
    
    Outputs
    -------
    summary: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Initialize separate counters for the batch and analysis stages.
    summary = {
        "runs": {"completed": 0, "skipped_existing": 0, "failed": 0},
        "analysis": {
            "completed": 0,
            "failed": 0,
            "skipped_missing_dependency": 0,
            "skipped_parent_failure": 0,
        },
    }

    # Tally the run statuses across the completed batch records.
    for record in run_records:
        status = record.get("status", "")
        # Only tally statuses that belong to the run-summary table.
        if status in summary["runs"]:
            summary["runs"][status] += 1

    # Tally the analysis statuses across the executed notebook records.
    for record in [] if analysis_records is None else analysis_records:
        status = record.get("status", "")
        # Only tally statuses that belong to the analysis-summary table.
        if status in summary["analysis"]:
            summary["analysis"][status] += 1

    # Emit a concise progress update for the current batch stage.
    print(
        "Runs:"
        f" completed={summary['runs']['completed']},"
        f" skipped_existing={summary['runs']['skipped_existing']},"
        f" failed={summary['runs']['failed']}"
    )

    # Print the analysis summary only when analysis records were supplied.
    if analysis_records is not None:
        # Emit a concise progress update for the current batch stage.
        print(
            "Analysis:"
            f" completed={summary['analysis']['completed']},"
            f" failed={summary['analysis']['failed']},"
            f" skipped_missing_dependency={summary['analysis']['skipped_missing_dependency']},"
            f" skipped_parent_failure={summary['analysis']['skipped_parent_failure']}"
        )

    # Return the compact status summary for the batch and analysis stages.
    return summary


CONFIG_FILE = None  # Set this to a config.py path to override automatic resolution.
ANALYSIS_NOTEBOOK = None  # Set this to a plots.ipynb path to override automatic resolution.


def load_comparison_gallery_module():
    '''Load the static comparison-gallery helper module.'''

    candidates = [
        Path('comparison_gallery.py'),
        Path('Code/Time-Distance/comparison_gallery.py'),
    ]
    for candidate in candidates:
        if candidate.exists():
            return load_local_module('comparison_gallery', candidate.resolve())

    raise FileNotFoundError('Could not find comparison_gallery.py.')


comparison_view = load_comparison_gallery_module()
write_default_comparison_viewers = comparison_view.write_default_comparison_viewers



In [ ]:
# Allow small height-grid mismatches for plotting-only comparisons
def infer_shared_single_cube_height_grid_km(time_distance_module: Any, cube_paths: Iterable[str | Path], *, rtol: float = 1.0e-9, atol: float = 1.0e-6) -> list[float]:
    """
    Compatibility override: accept tiny height-grid mismatches (warnings only)
    """
    import numpy as _np
    import warnings as _warnings

    # Normalize and deduplicate the requested cube paths before building the plan.
    normalized_paths = normalize_cube_paths(cube_paths)

    # Require at least one normalized cube path before continuing.
    if len(normalized_paths) == 0:
        raise ValueError("Need at least one single_cube path to infer a shared height grid.")

    # Use the first cube as the reference height grid for all subsequent comparisons.
    reference_path = normalized_paths[0]
    # Read the reference height coordinates from the first normalized cube.
    reference_heights = _np.asarray(
        time_distance_module.infer_netcdf_height_coordinates_km(reference_path),
        dtype = _np.float64,
    )

    # Iterate through the normalized cube paths so every case is handled once.
    for cube_path in normalized_paths[1:]:
        # Read the comparison cube height grid before checking it against the reference.
        comparison_heights = _np.asarray(
            time_distance_module.infer_netcdf_height_coordinates_km(cube_path),
            dtype = _np.float64,
        )

        # Fast equality check using caller-supplied tolerances.
        grids_match = (
            comparison_heights.shape == reference_heights.shape
            and _np.allclose(
                comparison_heights,
                reference_heights,
                rtol = float(rtol),
                atol = float(atol),
                equal_nan = True,
            )
        )

        if not grids_match:
            # Allow tiny mismatches for plotting-only comparisons by default
            # (e.g., small rounding/typing differences from vertical binning pipelines).
            if comparison_heights.shape == reference_heights.shape:
                max_abs = float(_np.max(_np.abs(comparison_heights - reference_heights)))
            else:
                max_abs = float('inf')

            # Treat differences <= 1e-3 km (~1 m) as acceptable for plotting.
            allowed_tol = max(float(atol), 1.0e-3)
            if _np.isfinite(max_abs) and max_abs <= allowed_tol:
                _warnings.warn(
                    f"Height grids differ slightly (max diff={max_abs:g} km); accepting for plotting.",
                    RuntimeWarning,
                    stacklevel = 2,
                )
            else:
                raise ValueError(
                    "All single_cube comparison paths must share the same height coordinate grid. "
                    f"{reference_path} and {cube_path} do not match."
                )

    # Return the assembled helper result for downstream batch logic.
    return [float(value) for value in reference_heights]


In [ ]:
# Observation Cubes Batch Cell
config_file, td, base_config = load_mode_base_config("paired_cubes", config_file = CONFIG_FILE)
paired_batch_settings = get_batch_mode_settings(config_file, "paired_cubes")

# Set the batch-run behavior for execution and follow-up analysis.
skip_existing = paired_batch_settings.get("skip_existing", True)
continue_on_error = paired_batch_settings.get("continue_on_error", True)
run_analysis = paired_batch_settings.get("run_analysis", True)
analysis_timeout = paired_batch_settings.get("analysis_timeout", 3600)

# Build the paired-cube plan directly from config.py batch_config.
paired_plan = build_configured_paired_cubes_batch_plan(base_config, td, paired_batch_settings)

v1_list = paired_plan["v1_list"]
v2_list = paired_plan["v2_list"]

# Preview the planned file pairs before running the pipeline.
print(f"Queued {len(paired_plan['run_configs'])} paired-cube runs.")
for index, (v1, v2) in enumerate(paired_plan["file_pairs"], start = 1):
    print(f"[{index:03d}] {Path(v1).parent.name}: {Path(v1).name} <> {Path(v2).name}")

# Execute the planned runs and print the saved outputs.
paired_run_records = execute_batch_runs(td, config_file, paired_plan["run_configs"], skip_existing = skip_existing, continue_on_error = continue_on_error)

# Optionally execute the analysis notebook for each completed run.
if run_analysis:
    paired_analysis_records = execute_td_analysis_for_batch(paired_run_records, analysis_notebook = ANALYSIS_NOTEBOOK, timeout = analysis_timeout, continue_on_error = continue_on_error)
else:
    paired_analysis_records = []

# Summarize the batch outcome for quick inspection in the notebook.
paired_summary = summarize_batch_records(paired_run_records, paired_analysis_records)

paired_summary


In [ ]:
# Simulation Batch Cell
config_file, td, base_config = load_mode_base_config("single_netcdf_cube", config_file = CONFIG_FILE)
single_batch_settings = get_batch_mode_settings(config_file, "single_netcdf_cube")
single_batch_inputs = resolve_single_cube_batch_inputs(base_config, single_batch_settings)

cube_paths = single_batch_inputs["cube_paths"]
observable = single_batch_inputs["observable"]
model_atmosphere_path = single_batch_inputs["model_atmosphere_path"]

# Set the batch-run behavior for execution and follow-up analysis.
skip_existing = single_batch_settings.get("skip_existing", True)
continue_on_error = single_batch_settings.get("continue_on_error", True)
run_analysis = single_batch_settings.get("run_analysis", True)
analysis_timeout = single_batch_settings.get("analysis_timeout", 3600)

# Build the list of single-cube runs from the shared settings.
single_plan = build_single_cube_batch_plan(base_config, td, cube_paths, observable = observable, model_atmosphere_path = model_atmosphere_path)

height_pairs_by_cube = single_plan["height_pairs_by_cube"]
height_pair_rows = single_plan["height_pair_rows"]

# Preview the planned height-pair coverage before running the pipeline.
print(f"Queued {len(single_plan['run_configs'])} single-cube runs.")
for cube_path, height_pairs in height_pairs_by_cube.items():
    print(f"{Path(cube_path).name}: {len(height_pairs)} height pairs")

# Execute the planned runs and print the saved outputs.
single_run_records = execute_batch_runs(td, config_file, single_plan["run_configs"], skip_existing = skip_existing, continue_on_error = continue_on_error)

# Optionally execute the analysis notebook for each completed run.
if run_analysis:
    single_analysis_records = execute_td_analysis_for_batch(single_run_records, analysis_notebook = ANALYSIS_NOTEBOOK, timeout = analysis_timeout, continue_on_error = continue_on_error)
else:
    single_analysis_records = []

# Summarize the batch outcome for quick inspection in the notebook.
single_summary = summarize_batch_records(single_run_records, single_analysis_records)

single_summary


In [ ]:
# Simulation Field-Strength Comparison Analysis Cell
config_file, td, base_config = load_mode_base_config("single_netcdf_cube", config_file = CONFIG_FILE)
single_batch_settings = get_batch_mode_settings(config_file, "single_netcdf_cube")
single_batch_inputs = resolve_single_cube_batch_inputs(base_config, single_batch_settings)

cube_paths = single_batch_inputs["cube_paths"]
observable = single_batch_inputs["observable"]
model_atmosphere_path = single_batch_inputs["model_atmosphere_path"]

# Set the execution behavior for the parent runs and comparison analysis.
skip_existing = single_batch_settings.get("skip_existing", True)
continue_on_error = single_batch_settings.get("continue_on_error", True)
run_time_distance_batch = single_batch_settings.get("run_time_distance_batch", True)
comparison_run_analysis = single_batch_settings.get("comparison_run_analysis", True)
comparison_timeout = single_batch_settings.get("comparison_timeout", 3600)

# Optionally regenerate the parent single-cube runs before plotting comparisons.
if run_time_distance_batch:
    comparison_single_plan = build_single_cube_batch_plan(base_config, td, cube_paths, observable = observable, model_atmosphere_path = model_atmosphere_path)
    comparison_parent_run_records = execute_batch_runs(td, config_file, comparison_single_plan["run_configs"], skip_existing = skip_existing, continue_on_error = continue_on_error)
else:
    comparison_parent_run_records = []

# Build the comparison requests from the configured cube set.
comparison_plan = build_single_cube_field_strength_comparison_plan(base_config, td, cube_paths, observable = observable, model_atmosphere_path = model_atmosphere_path)

# Preview the requested field-strength comparisons before execution.
print(f"Queued {len(comparison_plan['comparison_runs'])} field-strength comparison plots.")
for index, comparison_run in enumerate(comparison_plan['comparison_runs'], start = 1):
    print(
        f"[{index:03d}] "
        f"{comparison_run['h1_km']:g} km vs {comparison_run['h2_km']:g} km")

# Collect the notebook-execution status for each comparison request.
comparison_analysis_records = []

# Execute the analysis notebook for each comparison request when enabled.
if comparison_run_analysis:
    total_comparisons = len(comparison_plan['comparison_runs'])
    for comparison_index, comparison_run in enumerate(comparison_plan['comparison_runs'], start = 1):
        label = f"{comparison_run['h1_km']:g} km vs {comparison_run['h2_km']:g} km"
        print(f"[comparison {comparison_index}/{total_comparisons}] {label}")

        try:
            executed_notebook = execute_td_analysis_notebook(comparison_run['runtime_config'], analysis_notebook = ANALYSIS_NOTEBOOK, use_config = False, direct_plot_requests = comparison_run['direct_plot_requests'], timeout = comparison_timeout)
            comparison_analysis_records.append(
                {
                    'comparison_index': int(comparison_index),
                    'total_comparisons': int(total_comparisons),
                    'label': label,
                    'status': 'completed',
                    'executed_notebook': str(executed_notebook),
                }
            )
            print(f"  wrote {executed_notebook}")
        except Exception as exc:
            error_message = f"{exc.__class__.__name__}: {exc}"
            comparison_analysis_records.append(
                {
                    'comparison_index': int(comparison_index),
                    'total_comparisons': int(total_comparisons),
                    'label': label,
                    'status': 'failed',
                    'error': error_message,
                }
            )
            print(f"  failed: {error_message}")
            if not continue_on_error:
                raise

comparison_analysis_records


In [ ]:
# Simulation Cube Field-Orientation Comparison Analysis Cell
config_file, td, orientation_base_config = load_mode_base_config("single_netcdf_cube", config_file = CONFIG_FILE)
orientation_batch_settings = get_batch_mode_settings(config_file, "single_netcdf_cube")
orientation_batch_inputs = resolve_single_cube_batch_inputs(orientation_base_config, orientation_batch_settings)

orientation_cube_paths = orientation_batch_inputs["cube_paths"]
orientation_observable = orientation_batch_inputs["observable"]
orientation_model_atmosphere_path = orientation_batch_inputs["model_atmosphere_path"]

# Set the execution behavior for the parent runs and comparison analysis.
orientation_skip_existing = orientation_batch_settings.get("skip_existing", True)
orientation_continue_on_error = orientation_batch_settings.get("continue_on_error", True)
orientation_run_time_distance_batch = orientation_batch_settings.get("run_time_distance_batch", True)
orientation_comparison_run_analysis = orientation_batch_settings.get("comparison_run_analysis", True)
orientation_comparison_timeout = orientation_batch_settings.get("comparison_timeout", 3600)

# Optionally regenerate the parent single-cube runs before plotting comparisons.
if orientation_run_time_distance_batch:
    orientation_single_plan = build_single_cube_batch_plan(orientation_base_config, td, orientation_cube_paths, observable = orientation_observable, model_atmosphere_path = orientation_model_atmosphere_path)
    orientation_parent_run_records = execute_batch_runs(td, config_file, orientation_single_plan["run_configs"], skip_existing = orientation_skip_existing, continue_on_error = orientation_continue_on_error)
else:
    orientation_parent_run_records = []

# Build the comparison requests from the configured cube set.
orientation_comparison_plan = build_single_cube_field_orientation_comparison_plan(orientation_base_config, td, orientation_cube_paths, observable = orientation_observable, model_atmosphere_path = orientation_model_atmosphere_path)

# Preview the requested field-orientation comparisons before execution.
print(f"Queued {len(orientation_comparison_plan['comparison_runs'])} field-orientation comparison plots.")
for index, comparison_run in enumerate(orientation_comparison_plan['comparison_runs'], start = 1):
    print(
        f"[{index:03d}] "
        f"{comparison_run['h1_km']:g} km vs {comparison_run['h2_km']:g} km"
    )

# Collect the notebook-execution status for each comparison request.
orientation_comparison_analysis_records = []

# Execute the analysis notebook for each comparison request when enabled.
if orientation_comparison_run_analysis:
    total_comparisons = len(orientation_comparison_plan['comparison_runs'])
    for comparison_index, comparison_run in enumerate(orientation_comparison_plan['comparison_runs'], start = 1):
        label = f"{comparison_run['h1_km']:g} km vs {comparison_run['h2_km']:g} km"
        print(f"[comparison {comparison_index}/{total_comparisons}] {label}")

        try:
            executed_notebook = execute_td_analysis_notebook(comparison_run['runtime_config'], analysis_notebook = ANALYSIS_NOTEBOOK, use_config = False, direct_plot_requests = comparison_run['direct_plot_requests'], timeout = orientation_comparison_timeout)
            orientation_comparison_analysis_records.append(
                {
                    'comparison_index': int(comparison_index),
                    'total_comparisons': int(total_comparisons),
                    'label': label,
                    'status': 'completed',
                    'executed_notebook': str(executed_notebook),
                }
            )
            print(f"  wrote {executed_notebook}")
        except Exception as exc:
            error_message = f"{exc.__class__.__name__}: {exc}"
            orientation_comparison_analysis_records.append(
                {
                    'comparison_index': int(comparison_index),
                    'total_comparisons': int(total_comparisons),
                    'label': label,
                    'status': 'failed',
                    'error': error_message,
                }
            )
            print(f"  failed: {error_message}")
            if not orientation_continue_on_error:
                raise

orientation_comparison_analysis_records


In [ ]:
# Gaussian-Filter Comparison Analysis Cell

# Run both configured Gaussian comparison modes without manual path edits.
gaussian_comparison_modes = ["single_netcdf_cube", "paired_cubes"]

# Set the execution behavior for the parent runs and Gaussian-filter comparisons.
gaussian_comparison_skip_existing = True
gaussian_comparison_continue_on_error = True
gaussian_comparison_run_time_distance_batch = True
gaussian_comparison_run_analysis = True
gaussian_comparison_timeout = 5400

gaussian_comparison_records_by_mode = {}

for gaussian_comparison_mode in gaussian_comparison_modes:
    normalized_gaussian_mode = normalize_batch_source_type(gaussian_comparison_mode)
    config_file, td, gaussian_comparison_base_config = load_mode_base_config(normalized_gaussian_mode, config_file = CONFIG_FILE)
    gaussian_batch_settings = get_batch_mode_settings(config_file, gaussian_comparison_mode)
    gaussian_comparison_filter_params_list = resolve_batch_gaussian_filter_params(gaussian_comparison_base_config, gaussian_batch_settings)

    if normalized_gaussian_mode == "single_cube":
        single_batch_inputs = resolve_single_cube_batch_inputs(gaussian_comparison_base_config, gaussian_batch_settings)
        gaussian_comparison_plan = build_single_cube_gaussian_filter_comparison_plan(
            gaussian_comparison_base_config,
            td,
            single_batch_inputs["cube_paths"],
            gaussian_comparison_filter_params_list,
            observable = single_batch_inputs["observable"],
            model_atmosphere_path = single_batch_inputs["model_atmosphere_path"],
        )
        mode_label = "single_netcdf_cube"
        preview_label = "single-cube Gaussian-filter comparison"
    elif normalized_gaussian_mode == "paired_cubes":
        paired_defaults = gaussian_comparison_base_config["data"]["paired_cubes"]
        gaussian_pair_records = collect_paired_cube_batch_pair_records(
            gaussian_batch_settings,
            td,
            default_delta_z_km = float(gaussian_batch_settings.get("delta_z_km", paired_defaults["delta_z_km"])),
            default_p_dx_Mm = float(gaussian_batch_settings.get("p_dx_Mm", paired_defaults["p_dx_Mm"])),
            default_dt = float(gaussian_batch_settings.get("dt", paired_defaults["dt"])),
        )
        gaussian_comparison_plan = build_paired_cubes_gaussian_filter_comparison_plan(
            gaussian_comparison_base_config,
            td,
            gaussian_pair_records,
            gaussian_comparison_filter_params_list,
        )
        mode_label = "paired_cubes"
        preview_label = "paired-cubes Gaussian-filter comparison"
    else:
        raise ValueError(f"Unsupported Gaussian comparison mode: {gaussian_comparison_mode}")

    # Preview the requested Gaussian-filter comparisons before execution.
    print(
        f"Queued {len(gaussian_comparison_plan['comparison_runs'])} {preview_label} plot request(s) "
        f"from {len(gaussian_comparison_plan['run_configs'])} parent runs."
    )

    for index, comparison_run in enumerate(gaussian_comparison_plan['comparison_runs'], start = 1):
        if normalized_gaussian_mode == "single_cube":
            label = f"{comparison_run['h1_km']:g} km vs {comparison_run['h2_km']:g} km"
        else:
            label = f"{comparison_run['pair_count']} paired height permutations x {comparison_run['filter_count']} filters"
        print(f"[{index:03d}] {label}")

    # Optionally regenerate the parent runs before plotting comparisons.
    if gaussian_comparison_run_time_distance_batch:
        gaussian_comparison_parent_run_records = execute_batch_runs(
            td,
            config_file,
            gaussian_comparison_plan["run_configs"],
            skip_existing = gaussian_comparison_skip_existing,
            continue_on_error = gaussian_comparison_continue_on_error,
        )
    else:
        gaussian_comparison_parent_run_records = []

    # Collect the notebook-execution status for each comparison request.
    gaussian_comparison_analysis_records = []

    # Execute the analysis notebook for each comparison request when enabled.
    if gaussian_comparison_run_analysis:
        total_comparisons = len(gaussian_comparison_plan['comparison_runs'])
        for comparison_index, comparison_run in enumerate(gaussian_comparison_plan['comparison_runs'], start = 1):
            if normalized_gaussian_mode == "single_cube":
                label = f"{comparison_run['h1_km']:g} km vs {comparison_run['h2_km']:g} km"
            else:
                label = f"{comparison_run['pair_count']} paired height permutations x {comparison_run['filter_count']} filters"
            print(f"[{mode_label} comparison {comparison_index}/{total_comparisons}] {label}")

            try:
                executed_notebook = execute_td_analysis_notebook(
                    comparison_run['runtime_config'],
                    analysis_notebook = ANALYSIS_NOTEBOOK,
                    use_config = False,
                    direct_plot_requests = comparison_run['direct_plot_requests'],
                    timeout = gaussian_comparison_timeout,
                )
                gaussian_comparison_analysis_records.append(
                    {
                        'comparison_index': int(comparison_index),
                        'total_comparisons': int(total_comparisons),
                        'label': label,
                        'status': 'completed',
                        'executed_notebook': str(executed_notebook),
                    }
                )
                print(f"  wrote {executed_notebook}")
            except Exception as exc:
                error_message = f"{exc.__class__.__name__}: {exc}"
                gaussian_comparison_analysis_records.append(
                    {
                        'comparison_index': int(comparison_index),
                        'total_comparisons': int(total_comparisons),
                        'label': label,
                        'status': 'failed',
                        'error': error_message,
                    }
                )
                print(f"  failed: {error_message}")
                if not gaussian_comparison_continue_on_error:
                    raise

    gaussian_comparison_records_by_mode[mode_label] = {
        "plan": gaussian_comparison_plan,
        "parent_run_records": gaussian_comparison_parent_run_records,
        "analysis_records": gaussian_comparison_analysis_records,
    }

gaussian_comparison_records_by_mode


In [ ]:
"""Static HTML galleries for comparison-plot outputs.

This notebook cell inlines the former standalone gallery module so batch plots
can generate their HTML viewers without importing a separate module.
"""

from __future__ import annotations

from dataclasses import dataclass
from html import escape
import importlib.util
import json
from pathlib import Path
import re
import struct
from typing import Any, Iterable
from urllib.parse import quote


IMAGE_EXTENSIONS = (".jpeg", ".jpg", ".png")

COMPARISON_VIEWERS = {
    "field_strength_comparison": {
        "title": "Field Strength Comparison",
        "output_name": "field_strength_comparison.html",
    },
    "field_orientation_comparison": {
        "title": "Field Orientation Comparison",
        "output_name": "field_orientation_comparison.html",
    },
    "gaussian_filter_comparison": {
        "title": "Gaussian Filter Comparison",
        "output_name": "gaussian_filter_comparison.html",
    },
}

FULL_RESOLUTION_VIEW_DIRNAME = "comparison_view"

PANEL_ROWS = (
    ("komega", "k-omega diagram"),
    ("xc", "cross-correlation"),
    ("phase_diff", "phase difference"),
)

FIELD_STRENGTH_CASE_RE = re.compile(r"(z0|hx|vx)_((?:\d+(?:_\d+)?g)(?:_\d+(?:_\d+)?g)*)")
FIELD_ORIENTATION_CASE_RE = re.compile(r"([hv])(\d+(?:_\d+)?g)")
PROCESSING_MARKERS = (
    "_magnetogram_gaussian_filtered_",
    "_gaussian_magnetogram_filtered_",
    "_gaussian_filtered_",
    "_unfiltered",
)
XCORR_DIRECTIONAL_GEOMETRIES = {
    "east": "East Wedge",
    "west": "West Wedge",
    "north": "North Wedge",
    "south": "South Wedge",
}
_OUTPUT_PATHS_MODULE = None


def _load_output_paths_module():
    """Load the shared output-path helper without relying on sys.path state."""

    global _OUTPUT_PATHS_MODULE
    if _OUTPUT_PATHS_MODULE is not None:
        return _OUTPUT_PATHS_MODULE

    candidates = []
    if "__file__" in globals():
        candidates.append(Path(__file__).resolve().with_name("output_paths.py"))
    candidates.extend([
        Path("output_paths.py"),
        Path("Code/Time-Distance/output_paths.py"),
    ])
    module_path = next((candidate.expanduser().resolve() for candidate in candidates if candidate.expanduser().exists()), None)
    if module_path is None:
        return None

    spec = importlib.util.spec_from_file_location("comparison_gallery_output_paths", module_path)
    if spec is None or spec.loader is None:
        return None

    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    _OUTPUT_PATHS_MODULE = module
    return module


@dataclass(frozen=True)
class ComparisonPlot:
    """Metadata for one discovered comparison plot."""

    comparison_type: str
    path: Path
    relative_src: str
    h1_km: float
    h2_km: float
    width: int | None
    height: int | None
    xcorr_geometry: str = "annulus"

    @property
    def height_pair(self) -> tuple[float, float]:
        return (self.h1_km, self.h2_km)

    @property
    def geometry_label(self) -> str:
        return XCORR_DIRECTIONAL_GEOMETRIES.get(self.xcorr_geometry, "")


@dataclass(frozen=True)
class ComparisonPlotNameParts:
    """Parsed fields encoded in a comparison plot filename."""

    comparison_type: str
    observable: str
    h1_token: str
    h2_token: str
    tail: str


def format_height_km(value: float) -> str:
    """Format a height value for compact, deterministic labels."""

    if abs(value - round(value)) < 1.0e-9:
        return str(int(round(value)))

    return f"{value:g}"


def format_height_pair(height_pair: tuple[float, float]) -> str:
    """Return a human-readable height-pair label."""

    return f"{format_height_km(height_pair[0])}-{format_height_km(height_pair[1])} km"


def _parse_height_token(token: str) -> float:
    return float(token.replace("_", "."))


def _height_sort_key(height_pair: tuple[float, float]) -> tuple[float, float]:
    return (float(height_pair[0]), float(height_pair[1]))


def _comparison_filename_pattern(comparison_type: str) -> re.Pattern[str]:
    return re.compile(
        rf"^{re.escape(comparison_type)}_"
        r"(?P<observable>[a-zA-Z0-9]+)_"
        r"(?P<h1>\d+(?:_\d+)?)km_"
        r"(?P<h2>\d+(?:_\d+)?)km_"
    )


def read_image_size(image_path: Path) -> tuple[int, int] | tuple[None, None]:
    """Read PNG or JPEG dimensions without importing image-processing packages."""

    image_path = Path(image_path)
    try:
        with image_path.open("rb") as handle:
            header = handle.read(24)
            if header.startswith(b"\x89PNG\r\n\x1a\n") and header[12:16] == b"IHDR":
                width, height = struct.unpack(">II", header[16:24])
                return int(width), int(height)

            if header[:2] != b"\xff\xd8":
                return None, None

            handle.seek(2)
            while True:
                marker_prefix = handle.read(1)
                if marker_prefix == b"":
                    return None, None
                if marker_prefix != b"\xff":
                    continue

                marker = handle.read(1)
                while marker == b"\xff":
                    marker = handle.read(1)
                if marker == b"":
                    return None, None

                marker_code = marker[0]
                if marker_code in (0xD8, 0xD9):
                    continue

                segment_length_bytes = handle.read(2)
                if len(segment_length_bytes) != 2:
                    return None, None
                segment_length = struct.unpack(">H", segment_length_bytes)[0]
                if segment_length < 2:
                    return None, None

                if marker_code in {
                    0xC0,
                    0xC1,
                    0xC2,
                    0xC3,
                    0xC5,
                    0xC6,
                    0xC7,
                    0xC9,
                    0xCA,
                    0xCB,
                    0xCD,
                    0xCE,
                    0xCF,
                }:
                    segment = handle.read(segment_length - 2)
                    if len(segment) < 5:
                        return None, None
                    height, width = struct.unpack(">HH", segment[1:5])
                    return int(width), int(height)

                handle.seek(segment_length - 2, 1)
    except OSError:
        return None, None


def _parse_comparison_plot_name(plot: ComparisonPlot) -> ComparisonPlotNameParts | None:
    match = re.match(
        rf"^(?P<comparison_type>{'|'.join(re.escape(name) for name in COMPARISON_VIEWERS)})_"
        r"(?P<observable>[a-zA-Z0-9]+)_"
        r"(?P<h1>\d+(?:_\d+)?)km_"
        r"(?P<h2>\d+(?:_\d+)?)km_"
        r"(?P<tail>.+)$",
        plot.path.stem,
    )
    if match is None:
        return None

    return ComparisonPlotNameParts(
        comparison_type=match.group("comparison_type"),
        observable=match.group("observable"),
        h1_token=match.group("h1"),
        h2_token=match.group("h2"),
        tail=match.group("tail"),
    )


def _extract_xcorr_geometry_from_tail(tail: str) -> str:
    tail = str(tail).strip("_").lower()
    if tail == "":
        return "annulus"

    last_token = tail.rsplit("_", 1)[-1]
    if last_token in XCORR_DIRECTIONAL_GEOMETRIES:
        return last_token

    return "annulus"


def _strip_xcorr_geometry_suffix(value: str) -> str:
    value = str(value).strip("_")
    geometry = _extract_xcorr_geometry_from_tail(value)
    if geometry == "annulus":
        return value

    suffix = f"_{geometry}"
    return value[: -len(suffix)] if value.lower().endswith(suffix) else value


def _format_plot_height_label(plot: ComparisonPlot) -> str:
    height_label = format_height_pair(plot.height_pair)
    if plot.geometry_label == "":
        return height_label

    return f"{height_label} - {plot.geometry_label}"


def _split_case_and_processing_tail(tail: str) -> tuple[str, str]:
    marker_positions = [
        (tail.find(marker), marker)
        for marker in PROCESSING_MARKERS
        if tail.find(marker) >= 0
    ]
    if len(marker_positions) == 0:
        return tail, ""

    marker_index, marker = min(marker_positions, key=lambda item: item[0])
    return tail[:marker_index], tail[marker_index + 1 :]


def _parse_field_strength_cases(case_part: str) -> list[str]:
    cases: list[str] = []
    for match in FIELD_STRENGTH_CASE_RE.finditer(case_part):
        component = match.group(1)
        strength_tokens = re.findall(r"\d+(?:_\d+)?g", match.group(2))
        cases.extend(f"{component}_{strength_token}" for strength_token in strength_tokens)

    return cases


def _parse_field_orientation_cases(case_part: str) -> list[str]:
    component_by_orientation = {"h": "hx", "v": "vx"}
    return [
        f"{component_by_orientation[match.group(1)]}_{match.group(2)}"
        for match in FIELD_ORIENTATION_CASE_RE.finditer(case_part)
    ]


def _comparison_cases_and_processing(plot: ComparisonPlot) -> tuple[ComparisonPlotNameParts | None, list[str], str]:
    parts = _parse_comparison_plot_name(plot)
    if parts is None:
        return None, [], ""

    if parts.comparison_type == "field_strength_comparison":
        case_part, processing_slug = _split_case_and_processing_tail(parts.tail)
        return parts, _parse_field_strength_cases(case_part), processing_slug

    if parts.comparison_type == "field_orientation_comparison":
        case_part, processing_slug = _split_case_and_processing_tail(parts.tail)
        return parts, _parse_field_orientation_cases(case_part), processing_slug

    return parts, [], ""


def _quoted_parent_image_src(path: Path, figure_dir: Path | None = None) -> str:
    path = Path(path).expanduser().resolve()
    if figure_dir is None:
        return f"../{quote(path.name)}"

    figure_dir = Path(figure_dir).expanduser().resolve()
    try:
        relative_path = path.relative_to(figure_dir)
    except ValueError:
        relative_path = Path(path.name)

    return quote(f"../{relative_path.as_posix()}")


def _build_image_stem_index(figure_dir: Path) -> dict[str, list[Path]]:
    figure_dir = Path(figure_dir).expanduser().resolve()
    index: dict[str, list[Path]] = {}
    if not figure_dir.exists():
        return index

    for path in sorted(figure_dir.rglob("*"), key=lambda candidate: candidate.relative_to(figure_dir).as_posix()):
        if (
            path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
            and FULL_RESOLUTION_VIEW_DIRNAME not in path.relative_to(figure_dir).parts
        ):
            index.setdefault(path.stem, []).append(path)

    return index


def _clean_source_image_stem(stem: str) -> str:
    output_paths = _load_output_paths_module()
    if output_paths is None or not hasattr(output_paths, "clean_output_filename"):
        return stem

    return Path(output_paths.clean_output_filename(f"{stem}.jpeg")).stem


def _product_directory_for_panel(panel_kind: str) -> str:
    product_by_kind = {
        "komega": "komega_diagram",
        "xc": "time_distance",
        "phase_diff": "phase_difference",
    }
    product = product_by_kind.get(panel_kind, "")
    output_paths = _load_output_paths_module()
    if product == "" or output_paths is None or not hasattr(output_paths, "product_directory"):
        return {"komega": "komega", "xc": "xcorr", "phase_diff": "phase"}.get(panel_kind, "")

    return output_paths.product_directory(product)


def _candidate_source_image_stems(source_stems: str | Iterable[str], panel_kind: str) -> list[str]:
    if isinstance(source_stems, str):
        source_stems = [source_stems]

    suffixes_by_kind = {
        "komega": ("_komega",),
        "xc": ("_xc",),
        "phase_diff": ("_phase_diff",),
    }

    candidates: list[str] = []
    for source_stem in source_stems:
        for suffix in suffixes_by_kind.get(panel_kind, ()):
            raw_stem = f"{source_stem}{suffix}"
            for candidate in (raw_stem, _clean_source_image_stem(raw_stem)):
                if candidate not in candidates:
                    candidates.append(candidate)

    return candidates


def _find_image_by_stem(
    figure_dir: Path,
    stem: str,
    source_index: dict[str, list[Path]] | None = None,
    panel_kind: str = "",
) -> Path | None:
    if source_index is not None and stem in source_index:
        candidates = [
            candidate
            for candidate in source_index[stem]
            if _candidate_matches_panel(candidate, panel_kind)
        ]
        if len(candidates) == 0:
            return None
        return sorted(candidates, key=lambda candidate: _candidate_rank(candidate, panel_kind, stem))[0]

    for extension in IMAGE_EXTENSIONS:
        candidate = Path(figure_dir) / f"{stem}{extension}"
        if candidate.exists() and _candidate_matches_panel(candidate, panel_kind):
            return candidate
    return None


def _candidate_matches_panel(path: Path, panel_kind: str) -> bool:
    stem = path.stem.lower()
    if "comparison_" in stem or stem.endswith("_comparison"):
        return False
    if "magnetic_orientation_validation" in stem or "orientation_validation" in stem:
        return False
    if panel_kind == "":
        return True
    if panel_kind == "komega":
        return stem.endswith("_komega")
    if panel_kind == "xc":
        return stem.endswith("_xc")
    if panel_kind == "phase_diff":
        return stem.endswith("_phase_diff")

    return True


def _candidate_rank(path: Path, panel_kind: str, requested_stem: str) -> tuple[int, str]:
    parts = [part.lower() for part in path.parts]
    product_dir = _product_directory_for_panel(panel_kind)
    score = 0
    if product_dir != "" and product_dir in parts:
        score -= 20
    if path.stem == requested_stem:
        score -= 10
    if path.suffix.lower() == ".jpeg":
        score -= 1

    return score, path.as_posix()


def _find_source_panel_image(
    figure_dir: Path,
    source_stems: str | Iterable[str],
    panel_kind: str,
    source_index: dict[str, list[Path]] | None = None,
) -> Path | None:
    for source_stem in _candidate_source_image_stems(source_stems, panel_kind):
        source = _find_image_by_stem(
            figure_dir,
            source_stem,
            source_index=source_index,
            panel_kind=panel_kind,
        )
        if source is not None:
            return source

    return None


def _filter_slug_from_processing(processing_slug: str) -> str:
    processing_slug = _strip_xcorr_geometry_suffix(processing_slug)
    for prefix in (
        "magnetogram_gaussian_filtered_",
        "gaussian_magnetogram_filtered_",
        "gaussian_filtered_",
    ):
        if processing_slug.startswith(prefix):
            return processing_slug[len(prefix) :]
    return processing_slug


def _find_gaussian_filter_image(
    figure_dir: Path,
    case_prefix: str,
    parts: ComparisonPlotNameParts,
    processing_slug: str,
) -> Path | None:
    filter_slug = _filter_slug_from_processing(processing_slug)
    if filter_slug == "" or filter_slug == "unfiltered":
        return None

    xcorr_geometry = _extract_xcorr_geometry_from_tail(processing_slug)
    source_prefix = f"{case_prefix}_{parts.observable}_{parts.h1_token}km_{parts.h2_token}km_"
    required_fragment = f"_gaussian_filter_{filter_slug}"
    candidates = sorted(
        path
        for path in figure_dir.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
            and FULL_RESOLUTION_VIEW_DIRNAME not in path.relative_to(figure_dir).parts
            and path.stem.startswith(source_prefix)
            and required_fragment in path.stem
            and (
                xcorr_geometry == "annulus"
                or f"_{xcorr_geometry}_" in path.stem
                or path.stem.endswith(f"_{xcorr_geometry}")
            )
        )
    )

    return candidates[0] if len(candidates) > 0 else None


def _source_stem(case_prefix: str, parts: ComparisonPlotNameParts, processing_slug: str) -> str:
    base = f"{case_prefix}_{parts.observable}_{parts.h1_token}km_{parts.h2_token}km"
    return base if processing_slug == "" else f"{base}_{processing_slug}"


def _source_stems_for_panel(
    comparison_type: str,
    case_prefix: str,
    parts: ComparisonPlotNameParts,
    processing_slug: str,
    panel_kind: str,
) -> list[str]:
    active_stem = _source_stem(case_prefix, parts, processing_slug)
    if panel_kind != "komega":
        return [active_stem]
    if comparison_type == "gaussian_filter_comparison":
        return [active_stem]

    # Standard field comparison panels render the unfiltered k-omega product
    # while xcorr and phase panels render the active filtered runtime products.
    base_stem = _source_stem(case_prefix, parts, "")
    candidates = [f"{base_stem}_unfiltered", base_stem, active_stem]
    return list(dict.fromkeys(candidates))


def _relative_source_path(path: Path, figure_dir: Path) -> str:
    path = Path(path).expanduser().resolve()
    figure_dir = Path(figure_dir).expanduser().resolve()
    try:
        return path.relative_to(figure_dir).as_posix()
    except ValueError:
        return path.name


def _source_target_for_path(
    path: Path | None,
    label: str,
    figure_dir: Path,
    expected: str = "",
) -> dict[str, Any]:
    if path is None:
        return {
            "src": "",
            "sourcePath": "",
            "alt": label,
            "missing": True,
            "expected": expected,
        }

    return {
        "src": _quoted_parent_image_src(path, figure_dir),
        "sourcePath": _relative_source_path(path, figure_dir),
        "sourceFile": str(path.expanduser().resolve()),
        "alt": label,
        "missing": False,
        "expected": expected,
    }


def _fallback_bands(size: int, expected_count: int) -> list[tuple[int, int]]:
    if size <= 0 or expected_count <= 0:
        return []

    bands: list[tuple[int, int]] = []
    for index in range(expected_count):
        start = int(round(size*index/expected_count))
        stop = int(round(size*(index + 1)/expected_count)) - 1
        bands.append((max(0, start), max(start, min(size - 1, stop))))
    return bands


def _expand_segments_to_count(
    segments: list[tuple[int, int]],
    expected_count: int,
) -> list[tuple[int, int]]:
    if expected_count <= 0 or len(segments) >= expected_count:
        return segments[:expected_count]
    if len(segments) == 0:
        return segments

    expanded = list(segments)
    while len(expanded) < expected_count:
        widths = [stop - start + 1 for start, stop in expanded]
        widest_index = max(range(len(expanded)), key=lambda index: widths[index])
        widest_start, widest_stop = expanded.pop(widest_index)
        parts = min(expected_count - len(expanded), max(2, round(widths[widest_index]/max(1.0, min(widths)))))
        split_points = [
            int(round(widest_start + (widest_stop - widest_start + 1)*part/parts))
            for part in range(parts + 1)
        ]
        split_segments = [
            (split_points[part], max(split_points[part], split_points[part + 1] - 1))
            for part in range(parts)
        ]
        expanded[widest_index:widest_index] = split_segments

    return expanded[:expected_count]


def _projection_segments(
    mask: Any,
    axis: int,
    expected_count: int,
    minimum_span: float,
    density_fraction: float = 0.02,
) -> list[tuple[int, int]]:
    import numpy as np

    if axis == 0:
        projection = mask.sum(axis=0)
        threshold = mask.shape[0]*density_fraction
    else:
        projection = mask.sum(axis=1)
        threshold = mask.shape[1]*density_fraction

    indices = np.where(projection > threshold)[0]
    if indices.size == 0:
        return []

    segments: list[tuple[int, int]] = []
    start = int(indices[0])
    previous = int(indices[0])
    for index_value in indices[1:]:
        index_value = int(index_value)
        if index_value > previous + 3:
            if previous - start + 1 >= minimum_span:
                segments.append((start, previous))
            start = index_value
        previous = index_value

    if previous - start + 1 >= minimum_span:
        segments.append((start, previous))

    return _expand_segments_to_count(segments, expected_count)


def _load_image_mask(image_path: Path) -> tuple[Any | None, int, int]:
    try:
        import numpy as np
        from PIL import Image
    except ImportError:
        width, height = read_image_size(image_path)
        return None, int(width or 0), int(height or 0)

    try:
        with Image.open(image_path) as image:
            rgb = image.convert("RGB")
            array = np.asarray(rgb)
    except OSError:
        width, height = read_image_size(image_path)
        return None, int(width or 0), int(height or 0)

    mask = (array < 245).any(axis=2)
    height, width = mask.shape
    return mask, int(width), int(height)


def _detect_panel_bands(
    image_path: Path,
    expected_columns: int,
    expected_rows: int,
) -> tuple[list[tuple[int, int]], list[tuple[int, int]], int, int, Any | None]:
    mask, width, height = _load_image_mask(image_path)
    if width <= 0 or height <= 0:
        return [], [], width, height, mask

    if mask is None:
        return (
            _fallback_bands(width, expected_columns),
            _fallback_bands(height, expected_rows),
            width,
            height,
            mask,
        )

    minimum_y_span = max(2.0, height/max(1, expected_rows*4))
    y_bands = _projection_segments(mask, axis=1, expected_count=expected_rows, minimum_span=minimum_y_span)
    if len(y_bands) == 0:
        y_bands = _fallback_bands(height, expected_rows)

    minimum_x_span = max(2.0, width/max(1, expected_columns*4))
    x_source = mask
    if len(y_bands) > 0:
        first_y0, first_y1 = y_bands[0]
        x_source = mask[first_y0:first_y1 + 1, :]

    x_bands = _projection_segments(x_source, axis=0, expected_count=expected_columns, minimum_span=minimum_x_span)
    if len(x_bands) == 0:
        x_bands = _fallback_bands(width, expected_columns)

    return x_bands, y_bands, width, height, mask


def _row_x_bands(
    mask: Any | None,
    y_band: tuple[int, int],
    image_width: int,
    expected_columns: int,
) -> list[tuple[int, int]]:
    if mask is None:
        return _fallback_bands(image_width, expected_columns)

    minimum_x_span = max(2.0, image_width/max(1, expected_columns*4))
    if expected_columns == 1:
        minimum_x_span = max(2.0, image_width/20)
    y0, y1 = y_band
    row_mask = mask[y0:y1 + 1, :]
    row_bands = _projection_segments(
        row_mask,
        axis=0,
        expected_count=expected_columns,
        minimum_span=minimum_x_span,
    )
    return row_bands if len(row_bands) > 0 else _fallback_bands(image_width, expected_columns)


def _bounds_percent(
    x_band: tuple[int, int],
    y_band: tuple[int, int],
    image_width: int,
    image_height: int,
) -> dict[str, float]:
    x0, x1 = x_band
    y0, y1 = y_band
    return {
        "left": 100.0*x0/image_width if image_width > 0 else 0.0,
        "top": 100.0*y0/image_height if image_height > 0 else 0.0,
        "width": 100.0*(x1 - x0 + 1)/image_width if image_width > 0 else 0.0,
        "height": 100.0*(y1 - y0 + 1)/image_height if image_height > 0 else 0.0,
    }


def _bounds_pixels(x_band: tuple[int, int], y_band: tuple[int, int]) -> dict[str, int]:
    x0, x1 = x_band
    y0, y1 = y_band
    return {
        "left": int(x0),
        "top": int(y0),
        "width": int(x1 - x0 + 1),
        "height": int(y1 - y0 + 1),
    }


def _target_id(*parts: object) -> str:
    raw = "-".join(str(part) for part in parts if str(part) != "")
    safe = re.sub(r"[^a-zA-Z0-9_-]+", "-", raw.lower()).strip("-")
    return safe or "subplot"


def _build_target(
    target_id: str,
    label: str,
    source: dict[str, Any],
    x_band: tuple[int, int],
    y_band: tuple[int, int],
    image_width: int,
    image_height: int,
) -> dict[str, Any]:
    source = dict(source)
    crop_bounds = _bounds_pixels(x_band, y_band)

    return {
        "id": target_id,
        "label": label,
        "src": source.get("src", ""),
        "sourcePath": source.get("sourcePath", ""),
        "sourceFile": source.get("sourceFile", ""),
        "alt": source.get("alt", label),
        "crop": source.get("crop", None),
        "bounds": _bounds_percent(x_band, y_band, image_width, image_height),
        "pixelBounds": crop_bounds,
        "missing": bool(source.get("missing", source.get("src", "") == "")),
        "expected": source.get("expected", ""),
    }


def _expected_panel_description(source_stems: str | Iterable[str], panel_kind: str) -> str:
    candidates = _candidate_source_image_stems(source_stems, panel_kind)
    return ", ".join(candidates)


def _build_field_comparison_targets(
    plot: ComparisonPlot,
    title: str,
    height_label: str,
    figure_dir: Path | None = None,
) -> list[dict[str, Any]]:
    parts, cases, processing_slug = _comparison_cases_and_processing(plot)
    if parts is None or len(cases) == 0:
        return []

    expected_rows = 4
    expected_columns = len(cases)
    x_bands, y_bands, width, height, mask = _detect_panel_bands(plot.path, expected_columns, expected_rows)
    if len(x_bands) < expected_columns or len(y_bands) < 3 or width <= 0 or height <= 0:
        return []

    figure_dir = Path(plot.path.parent if figure_dir is None else figure_dir).expanduser().resolve()
    source_index = _build_image_stem_index(figure_dir)
    targets: list[dict[str, Any]] = []

    for column_index, case_prefix in enumerate(cases):
        for row_index, (panel_kind, panel_label) in enumerate(PANEL_ROWS):
            source_stems = _source_stems_for_panel(plot.comparison_type, case_prefix, parts, processing_slug, panel_kind)
            source = _source_target_for_path(
                _find_source_panel_image(
                    figure_dir,
                    source_stems,
                    panel_kind,
                    source_index=source_index,
                ),
                f"{case_prefix} {panel_label}",
                figure_dir,
                expected=_expected_panel_description(source_stems, panel_kind),
            )
            targets.append(
                _build_target(
                    _target_id(case_prefix, panel_kind),
                    f"{case_prefix} {panel_label}",
                    source,
                    x_bands[column_index],
                    y_bands[row_index],
                    width,
                    height,
                )
            )

    if len(y_bands) >= 4:
        filter_x_bands = _row_x_bands(mask, y_bands[3], width, 1)
        if len(filter_x_bands) > 0:
            reference_case = cases[0]
            source = _source_target_for_path(
                _find_gaussian_filter_image(figure_dir, reference_case, parts, processing_slug),
                f"{title}: {height_label} Gaussian filter",
                figure_dir,
                expected=f"{reference_case}_{parts.observable}_{parts.h1_token}km_{parts.h2_token}km_*gaussian_filter_{_filter_slug_from_processing(processing_slug)}",
            )
            targets.append(
                _build_target(
                    _target_id("gaussian-filter", reference_case),
                    f"{title}: {height_label} Gaussian filter",
                    source,
                    filter_x_bands[0],
                    y_bands[3],
                    width,
                    height,
                )
            )

    return targets


def _extract_gaussian_filter_count(parts: ComparisonPlotNameParts) -> int:
    match = re.search(r"(?:^|_)filters_(\d+)(?:_|$)", parts.tail)
    return int(match.group(1)) if match is not None else 0


def _parse_gaussian_processing_slugs_from_tail(parts: ComparisonPlotNameParts) -> list[str]:
    matches = []
    for match in re.finditer(
        r"(?:^|_)f(?P<index>\d+)_"
        r"(?P<slug>gauss_ck_\d+(?:_\d+)?_wk_\d+(?:_\d+)?_cf_\d+(?:_\d+)?_wf_\d+(?:_\d+)?)"
        r"(?=_f\d+_|_|$)",
        parts.tail,
        flags=re.IGNORECASE,
    ):
        matches.append((int(match.group("index")), f"gaussian_filtered_{match.group('slug').lower()}"))

    return [slug for _, slug in sorted(matches)]


def _discover_gaussian_processing_slugs(
    figure_dir: Path,
    reference_case: str,
    parts: ComparisonPlotNameParts,
    expected_count: int,
) -> list[str]:
    parsed_slugs = _parse_gaussian_processing_slugs_from_tail(parts)
    if len(parsed_slugs) >= expected_count:
        return parsed_slugs[:expected_count]

    source_prefix = f"{reference_case}_{parts.observable}_{parts.h1_token}km_{parts.h2_token}km_"
    komega_suffix = "_komega_magnetic_orientation_validation"
    processing_slugs = []

    for path in sorted(figure_dir.rglob("*"), key=lambda candidate: candidate.relative_to(figure_dir).as_posix()):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        if FULL_RESOLUTION_VIEW_DIRNAME in path.relative_to(figure_dir).parts:
            continue
        if not path.stem.startswith(source_prefix) or not path.stem.endswith(komega_suffix):
            continue

        processing_slug = path.stem[len(source_prefix) : -len(komega_suffix)]
        if processing_slug == "unfiltered":
            continue
        processing_slugs.append(processing_slug)

    filter_tail = parts.tail
    ranked: list[tuple[int, str]] = []
    unranked: list[str] = []
    for processing_slug in processing_slugs:
        filter_slug = _filter_slug_from_processing(processing_slug)
        position = filter_tail.find(filter_slug)
        if position >= 0:
            ranked.append((position, processing_slug))
        else:
            unranked.append(processing_slug)

    if len(ranked) == 0:
        discovered_slugs = processing_slugs[:expected_count]
    else:
        discovered_slugs = ([slug for _, slug in sorted(ranked)] + sorted(unranked))[:expected_count]

    combined_slugs = parsed_slugs + [slug for slug in discovered_slugs if slug not in parsed_slugs]

    return combined_slugs[:expected_count]


def _build_gaussian_comparison_targets(
    plot: ComparisonPlot,
    title: str,
    height_label: str,
    figure_dir: Path | None = None,
) -> list[dict[str, Any]]:
    parts = _parse_comparison_plot_name(plot)
    if parts is None:
        return []

    case_prefixes = ["z0_0g", "hx_10g", "hx_50g", "hx_100g", "vx_10g", "vx_50g", "vx_100g"]
    filter_count = _extract_gaussian_filter_count(parts)
    if filter_count <= 0:
        return []

    expected_rows = 1 + len(case_prefixes)*len(PANEL_ROWS)
    x_bands, y_bands, width, height, _ = _detect_panel_bands(plot.path, filter_count, expected_rows)
    if len(x_bands) < filter_count or len(y_bands) < expected_rows or width <= 0 or height <= 0:
        return []

    figure_dir = Path(plot.path.parent if figure_dir is None else figure_dir).expanduser().resolve()
    source_index = _build_image_stem_index(figure_dir)
    processing_slugs = _discover_gaussian_processing_slugs(
        figure_dir,
        case_prefixes[0],
        parts,
        filter_count,
    )
    if len(processing_slugs) < filter_count:
        processing_slugs.extend([""]*(filter_count - len(processing_slugs)))
    processing_slugs = processing_slugs[:filter_count]

    targets: list[dict[str, Any]] = []
    for filter_index, processing_slug in enumerate(processing_slugs):
        source = _source_target_for_path(
            _find_gaussian_filter_image(figure_dir, case_prefixes[0], parts, processing_slug),
            f"{title}: {height_label} filter {filter_index + 1}",
            figure_dir,
            expected=f"{case_prefixes[0]}_{parts.observable}_{parts.h1_token}km_{parts.h2_token}km_*gaussian_filter_{_filter_slug_from_processing(processing_slug)}",
        )
        targets.append(
            _build_target(
                _target_id("filter", filter_index + 1),
                f"{title}: {height_label} filter {filter_index + 1}",
                source,
                x_bands[filter_index],
                y_bands[0],
                width,
                height,
            )
        )

    for case_index, case_prefix in enumerate(case_prefixes):
        for row_index, (panel_kind, panel_label) in enumerate(PANEL_ROWS):
            y_band = y_bands[1 + case_index*len(PANEL_ROWS) + row_index]
            for filter_index, processing_slug in enumerate(processing_slugs):
                source_stems = _source_stems_for_panel(plot.comparison_type, case_prefix, parts, processing_slug, panel_kind)
                source = _source_target_for_path(
                    _find_source_panel_image(
                        figure_dir,
                        source_stems,
                        panel_kind,
                        source_index=source_index,
                    ),
                    f"{case_prefix} filter {filter_index + 1} {panel_label}",
                    figure_dir,
                    expected=_expected_panel_description(source_stems, panel_kind),
                )
                targets.append(
                    _build_target(
                        _target_id(case_prefix, "filter", filter_index + 1, panel_kind),
                        f"{case_prefix} filter {filter_index + 1} {panel_label}",
                        source,
                        x_bands[filter_index],
                        y_band,
                        width,
                        height,
                    )
                )

    return targets


def build_subplot_navigation_targets(
    plot: ComparisonPlot,
    title: str,
    height_label: str,
    figure_dir: Path | None = None,
) -> list[dict[str, Any]]:
    """Build clickable subplot targets for a comparison view page."""

    if plot.comparison_type in {"field_strength_comparison", "field_orientation_comparison"}:
        return _build_field_comparison_targets(plot, title, height_label, figure_dir=figure_dir)
    if plot.comparison_type == "gaussian_filter_comparison":
        return _build_gaussian_comparison_targets(plot, title, height_label, figure_dir=figure_dir)
    return []


def resolved_subplot_navigation_targets(targets: Iterable[dict[str, Any]]) -> list[dict[str, Any]]:
    return [target for target in targets if not target.get("missing", False) and str(target.get("src", "")) != ""]


def _public_navigation_target(target: dict[str, Any]) -> dict[str, Any]:
    return {
        key: value
        for key, value in target.items()
        if key not in {"sourceFile", "missing", "expected"}
    }


def _target_panel_kind(target: dict[str, Any]) -> str:
    target_id = str(target.get("id", ""))
    if target_id.endswith("-komega"):
        return "komega"
    if target_id.endswith("-xc"):
        return "xc"
    if target_id.endswith("-phase_diff"):
        return "phase_diff"
    if target_id.startswith("filter-") or "gaussian-filter" in target_id:
        return "gaussian_filter"

    return ""


def _target_matches_declared_kind(target: dict[str, Any]) -> bool:
    source_path = str(target.get("sourcePath", "")).lower()
    panel_kind = _target_panel_kind(target)
    if source_path == "" or panel_kind == "":
        return True
    if "comparison_" in source_path:
        return False
    if "orientation_validation" in source_path or "magnetic_orientation_validation" in source_path:
        return False
    if panel_kind == "komega":
        return Path(source_path).stem.endswith("_komega")
    if panel_kind == "xc":
        return Path(source_path).stem.endswith("_xc")
    if panel_kind == "phase_diff":
        return Path(source_path).stem.endswith("_phase_diff")
    if panel_kind == "gaussian_filter":
        return "_gaussian_filter_" in Path(source_path).stem

    return True


def validate_subplot_navigation_targets(
    plot: ComparisonPlot,
    targets: Iterable[dict[str, Any]],
    figure_dir: Path,
) -> list[str]:
    figure_dir = Path(figure_dir).expanduser().resolve()
    warnings: list[str] = []
    for target in targets:
        if target.get("missing", False) or str(target.get("src", "")) == "":
            warnings.append(
                "missing target\t"
                f"comparison={plot.path.name}\t"
                f"target={target.get('id', '')}\t"
                f"label={target.get('label', '')}\t"
                f"expected={target.get('expected', '')}"
            )
            continue

        source_file = str(target.get("sourceFile", ""))
        if source_file == "":
            source_path = Path(str(target.get("sourcePath", "")))
            source_file = str((figure_dir / source_path).resolve())
        if not Path(source_file).exists():
            warnings.append(
                "missing file\t"
                f"comparison={plot.path.name}\t"
                f"target={target.get('id', '')}\t"
                f"source={target.get('sourcePath', '')}"
            )
        if not _target_matches_declared_kind(target):
            warnings.append(
                "product mismatch\t"
                f"comparison={plot.path.name}\t"
                f"target={target.get('id', '')}\t"
                f"source={target.get('sourcePath', '')}\t"
                f"expected={target.get('expected', '')}"
            )

    return warnings


def _json_script_data(data: object) -> str:
    return json.dumps(data, ensure_ascii=True, separators=(",", ":")).replace("</", "<\\/")


def discover_comparison_plots(
    figure_dir: str | Path,
    comparison_types: Iterable[str] | None = None,
) -> list[ComparisonPlot]:
    """Discover comparison plot images in deterministic filename order."""

    figure_dir = Path(figure_dir).expanduser().resolve()
    selected_types = list(COMPARISON_VIEWERS if comparison_types is None else comparison_types)
    patterns = {
        comparison_type: _comparison_filename_pattern(comparison_type)
        for comparison_type in selected_types
    }

    if not figure_dir.exists():
        return []

    plots: list[ComparisonPlot] = []
    candidates = sorted(
        (
            path
            for path in figure_dir.rglob("*")
            if path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
            and FULL_RESOLUTION_VIEW_DIRNAME not in path.relative_to(figure_dir).parts
        ),
        key=lambda path: path.relative_to(figure_dir).as_posix(),
    )

    for path in candidates:
        for comparison_type, pattern in patterns.items():
            match = pattern.match(path.name)
            if match is None:
                continue

            width, height = read_image_size(path)
            tail = path.stem[match.end() :]
            plots.append(
                ComparisonPlot(
                    comparison_type=comparison_type,
                    path=path,
                    relative_src=quote(path.relative_to(figure_dir).as_posix()),
                    h1_km=_parse_height_token(match.group("h1")),
                    h2_km=_parse_height_token(match.group("h2")),
                    width=width,
                    height=height,
                    xcorr_geometry=_extract_xcorr_geometry_from_tail(tail),
                )
            )
            break

    return sorted(
        plots,
        key=lambda plot: (
            plot.comparison_type,
            _height_sort_key(plot.height_pair),
            plot.path.name,
        ),
    )


def comparison_height_pairs(plots: Iterable[ComparisonPlot]) -> list[tuple[float, float]]:
    """Return the shared deterministic height-pair order for all galleries."""

    height_pairs = {plot.height_pair for plot in plots}
    return sorted(height_pairs, key=_height_sort_key)


def _dimension_attributes(plot: ComparisonPlot) -> str:
    if plot.width is None or plot.height is None:
        return ""

    return f' width="{plot.width}" height="{plot.height}"'


def _full_resolution_view_name(plot: ComparisonPlot) -> str:
    return f"{plot.path.stem}.html"


def _full_resolution_view_href(plot: ComparisonPlot) -> str:
    return f"{FULL_RESOLUTION_VIEW_DIRNAME}/{quote(_full_resolution_view_name(plot))}"


def _render_plot_link(plot: ComparisonPlot, title: str, height_label: str) -> str:
    src = escape(plot.relative_src, quote=True)
    view_href = escape(_full_resolution_view_href(plot), quote=True)
    image_title = escape(f"{title}: {height_label}", quote=True)
    dimension_attributes = _dimension_attributes(plot)

    return (
        f'<a class="figure-link" href="{view_href}" title="{image_title}" '
        f'data-source-path="{src}">\n'
        f'  <span class="height-label">{escape(height_label)}</span>\n'
        f'  <img src="{src}" alt="{image_title}"{dimension_attributes} '
        f'loading="lazy" decoding="async">\n'
        f"</a>"
    )


def _full_resolution_dimension_attributes(plot: ComparisonPlot) -> str:
    attributes = []
    if plot.width is not None:
        attributes.append(f' width="{plot.width}"')
    if plot.height is not None:
        attributes.append(f' height="{plot.height}"')
    return "".join(attributes)


def _ordered_gallery_plots(
    selected_plots: Iterable[ComparisonPlot],
    height_pairs: Iterable[tuple[float, float]],
) -> list[ComparisonPlot]:
    plots_by_height_pair: dict[tuple[float, float], list[ComparisonPlot]] = {}
    for plot in selected_plots:
        plots_by_height_pair.setdefault(plot.height_pair, []).append(plot)

    ordered_plots: list[ComparisonPlot] = []
    ordered_height_pairs = list(height_pairs)
    if len(ordered_height_pairs) == 0:
        ordered_height_pairs = comparison_height_pairs(selected_plots)

    for height_pair in ordered_height_pairs:
        ordered_plots.extend(sorted(plots_by_height_pair.get(height_pair, []), key=lambda plot: plot.path.name))

    return ordered_plots


def render_full_resolution_view_html(
    plot: ComparisonPlot,
    title: str,
    height_label: str,
    figure_dir: Path | None = None,
    navigation_targets: Iterable[dict[str, Any]] | None = None,
) -> str:
    """Render a minimal, borderless full-resolution inspection page."""

    figure_dir = Path(plot.path.parent if figure_dir is None else figure_dir).expanduser().resolve()
    src = escape(_quoted_parent_image_src(plot.path, figure_dir), quote=True)
    alt_text = escape(f"{title}: {height_label}", quote=True)
    dimension_attributes = _full_resolution_dimension_attributes(plot)
    frame_style = ""
    if plot.width is not None and plot.height is not None:
        frame_style = f' style="width: {plot.width}px; height: {plot.height}px;"'
    if navigation_targets is None:
        navigation_targets = build_subplot_navigation_targets(plot, title, height_label, figure_dir=figure_dir)
    navigation_targets = resolved_subplot_navigation_targets(navigation_targets)
    navigation_targets = [_public_navigation_target(target) for target in navigation_targets]
    rendered_hotspots = "\n".join(
        (
            f'    <a class="subplot-hotspot" href="{escape(str(target["src"]), quote=True)}" '
            f'data-nav-target="{escape(str(target["id"]), quote=True)}" '
            f'data-source-path="{escape(str(target.get("sourcePath", "")), quote=True)}" '
            f'title="{escape(str(target["label"]), quote=True)}" '
            f'aria-label="{escape(str(target["label"]), quote=True)}" '
            f'style="left: {target["bounds"]["left"]:.6f}%; '
            f'top: {target["bounds"]["top"]:.6f}%; '
            f'width: {target["bounds"]["width"]:.6f}%; '
            f'height: {target["bounds"]["height"]:.6f}%;"></a>'
        )
        for target in navigation_targets
    )
    targets_json = _json_script_data(navigation_targets)
    overview_json = _json_script_data(
        {
            "id": "overview",
            "label": f"{title}: {height_label}",
            "src": _quoted_parent_image_src(plot.path, figure_dir),
            "sourcePath": plot.path.name,
            "alt": f"{title}: {height_label}",
            "crop": None,
        }
    )

    return f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>{escape(title)} {escape(height_label)}</title>
  <style>
    html,
    body {{
      margin: 0;
      padding: 0;
      background: #ffffff;
    }}

    body {{
      min-height: 100vh;
    }}

    .figure-frame {{
      position: relative;
      display: block;
      overflow: visible;
    }}

    .figure-frame.is-managed img {{
      position: absolute;
      left: 0;
      top: 0;
      max-width: none;
      max-height: none;
    }}

    .figure-frame.is-crop {{
      overflow: hidden;
    }}

    img {{
      display: block;
      width: auto;
      height: auto;
      max-width: none;
      max-height: none;
      image-rendering: auto;
    }}

    .subplot-map {{
      position: absolute;
      inset: 0;
    }}

    .figure-frame:not(.is-overview) .subplot-map {{
      display: none;
    }}

    .subplot-hotspot {{
      position: absolute;
      display: block;
      cursor: pointer;
      background: rgba(0, 0, 0, 0);
      touch-action: pan-x pan-y;
    }}

    .subplot-hotspot:focus-visible {{
      outline: 2px solid #0b5cab;
      outline-offset: 2px;
    }}
  </style>
</head>
<body>
  <div class="figure-frame is-overview"{frame_style} data-figure-frame>
    <img data-full-resolution-image src="{src}" alt="{alt_text}"{dimension_attributes}>
    <div class="subplot-map" aria-label="{escape(title)} subplot navigation">
{rendered_hotspots}
    </div>
  </div>
  <script>
    (() => {{
      const overview = {overview_json};
      const targets = {targets_json};
      const targetById = new Map(targets.map((target) => [target.id, target]));
      const frame = document.querySelector("[data-figure-frame]");
      const image = document.querySelector("[data-full-resolution-image]");
      const minimumScale = 0.01;
      let baseWidth = Number(image.getAttribute("width")) || 0;
      let baseHeight = Number(image.getAttribute("height")) || 0;
      let scale = 1;
      let gestureStartScale = 1;
      let currentViewId = "overview";
      let pendingViewState = null;
      let pointerStart = null;

      const storageKey = () => `comparison-view:${{window.location.pathname}}:${{currentViewId}}`;

      const saveViewState = () => {{
        try {{
          window.sessionStorage.setItem(storageKey(), JSON.stringify({{
            scale,
            scrollX: window.scrollX,
            scrollY: window.scrollY,
          }}));
        }} catch (error) {{
          // Session storage can be unavailable for local files in some browsers.
        }}
      }};

      const loadViewState = (viewId) => {{
        try {{
          const value = window.sessionStorage.getItem(`comparison-view:${{window.location.pathname}}:${{viewId}}`);
          return value === null ? null : JSON.parse(value);
        }} catch (error) {{
          return null;
        }}
      }};

      const refreshBaseDimensions = () => {{
        baseWidth = image.naturalWidth || baseWidth || image.getBoundingClientRect().width;
        baseHeight = image.naturalHeight || baseHeight || image.getBoundingClientRect().height;
      }};

      const activeView = () => targetById.get(currentViewId) || overview;

      const setFrameGeometry = () => {{
        const view = activeView();
        const crop = view.crop || null;
        const visibleWidth = crop === null ? baseWidth : crop.width;
        const visibleHeight = crop === null ? baseHeight : crop.height;
        const renderedWidth = visibleWidth * scale;
        const renderedHeight = visibleHeight * scale;

        frame.style.width = `${{renderedWidth}}px`;
        frame.style.height = `${{renderedHeight}}px`;
        image.style.width = `${{baseWidth * scale}}px`;
        image.style.height = `${{baseHeight * scale}}px`;
        image.style.left = crop === null ? "0px" : `${{-crop.left * scale}}px`;
        image.style.top = crop === null ? "0px" : `${{-crop.top * scale}}px`;
        frame.classList.toggle("is-overview", currentViewId === "overview");
        frame.classList.toggle("is-crop", crop !== null);
      }};

      const applyScale = (nextScale, anchorX, anchorY) => {{
        if (!Number.isFinite(nextScale) || nextScale <= 0) {{
          return;
        }}

        refreshBaseDimensions();
        if (baseWidth <= 0 || baseHeight <= 0) {{
          return;
        }}

        const before = frame.getBoundingClientRect();
        const viewportX = Number.isFinite(anchorX) ? anchorX : window.innerWidth / 2;
        const viewportY = Number.isFinite(anchorY) ? anchorY : window.innerHeight / 2;
        const imageX = before.width > 0 ? (viewportX - before.left) / before.width : 0.5;
        const imageY = before.height > 0 ? (viewportY - before.top) / before.height : 0.5;
        const boundedScale = Math.max(minimumScale, nextScale);
        const view = activeView();
        const crop = view.crop || null;
        const visibleWidth = crop === null ? baseWidth : crop.width;
        const renderedWidth = visibleWidth * boundedScale;

        if (!Number.isFinite(renderedWidth) || renderedWidth <= 0) {{
          return;
        }}

        scale = boundedScale;
        setFrameGeometry();

        const after = frame.getBoundingClientRect();
        const targetLeft = window.scrollX + after.left + imageX * after.width - viewportX;
        const targetTop = window.scrollY + after.top + imageY * after.height - viewportY;
        window.scrollTo(targetLeft, targetTop);
      }};

      const applyStoredState = () => {{
        if (pendingViewState !== null && Number.isFinite(pendingViewState.scale)) {{
          scale = Math.max(minimumScale, pendingViewState.scale);
        }}
        setFrameGeometry();

        if (pendingViewState !== null) {{
          window.scrollTo(
            Number(pendingViewState.scrollX) || 0,
            Number(pendingViewState.scrollY) || 0,
          );
        }}

        pendingViewState = null;
      }};

      const showView = (viewId, updateHistory) => {{
        const nextView = viewId === "overview" ? overview : targetById.get(viewId);
        if (nextView === undefined) {{
          return;
        }}

        saveViewState();
        currentViewId = viewId;
        pendingViewState = loadViewState(viewId);
        scale = pendingViewState === null || !Number.isFinite(pendingViewState.scale)
          ? 1
          : Math.max(minimumScale, pendingViewState.scale);

        if (updateHistory) {{
          const nextUrl = viewId === "overview"
            ? window.location.pathname + window.location.search
            : `#subplot=${{encodeURIComponent(viewId)}}`;
          window.history.pushState({{ viewId }}, "", nextUrl);
        }}

        document.title = nextView.label || overview.label;
        image.alt = nextView.alt || nextView.label || overview.alt;
        if (image.getAttribute("src") !== nextView.src) {{
          image.addEventListener("load", () => {{
            refreshBaseDimensions();
            applyStoredState();
          }}, {{ once: true }});
          image.src = nextView.src;
        }} else {{
          refreshBaseDimensions();
          applyStoredState();
        }}
      }};

      const viewIdFromLocation = () => {{
        const match = window.location.hash.match(/^#subplot=(.+)$/);
        if (match === null) {{
          return "overview";
        }}

        const viewId = decodeURIComponent(match[1]);
        return targetById.has(viewId) ? viewId : "overview";
      }};

      frame.classList.add("is-managed");
      image.addEventListener("load", () => {{
        refreshBaseDimensions();
        setFrameGeometry();
      }}, {{ once: true }});
      refreshBaseDimensions();
      setFrameGeometry();
      currentViewId = viewIdFromLocation();
      window.history.replaceState({{ viewId: currentViewId }}, "", window.location.href);
      if (currentViewId !== "overview") {{
        showView(currentViewId, false);
      }}

      document.addEventListener("pointerdown", (event) => {{
        const link = event.target.closest("[data-nav-target]");
        pointerStart = link === null ? null : {{
          x: event.clientX,
          y: event.clientY,
          id: link.getAttribute("data-nav-target"),
        }};
      }}, true);

      document.addEventListener("click", (event) => {{
        const link = event.target.closest("[data-nav-target]");
        if (link === null) {{
          return;
        }}

        event.preventDefault();
        const viewId = link.getAttribute("data-nav-target");
        if (pointerStart !== null && pointerStart.id === viewId) {{
          const dx = event.clientX - pointerStart.x;
          const dy = event.clientY - pointerStart.y;
          if (Math.hypot(dx, dy) > 6) {{
            pointerStart = null;
            return;
          }}
        }}

        pointerStart = null;
        showView(viewId, true);
      }});

      window.addEventListener("popstate", (event) => {{
        const viewId = event.state && typeof event.state.viewId === "string"
          ? event.state.viewId
          : viewIdFromLocation();
        showView(viewId, false);
      }});

      window.addEventListener("pagehide", saveViewState);

      window.addEventListener("wheel", (event) => {{
        if (!event.ctrlKey && !event.metaKey) {{
          return;
        }}

        event.preventDefault();
        applyScale(scale * Math.exp(-event.deltaY * 0.002), event.clientX, event.clientY);
      }}, {{ passive: false }});

      window.addEventListener("gesturestart", (event) => {{
        event.preventDefault();
        gestureStartScale = scale;
      }}, {{ passive: false }});

      window.addEventListener("gesturechange", (event) => {{
        event.preventDefault();
        applyScale(gestureStartScale * event.scale, window.innerWidth / 2, window.innerHeight / 2);
      }}, {{ passive: false }});
    }})();
  </script>
</body>
</html>
"""


def render_comparison_view_html(
    comparison_type: str,
    plots: Iterable[ComparisonPlot],
    height_pairs: Iterable[tuple[float, float]],
) -> str:
    """Render one standalone comparison gallery."""

    if comparison_type not in COMPARISON_VIEWERS:
        raise ValueError(f"Unsupported comparison gallery type: {comparison_type}")

    viewer_config = COMPARISON_VIEWERS[comparison_type]
    title = viewer_config["title"]
    selected_plots = [plot for plot in plots if plot.comparison_type == comparison_type]
    ordered_plots = _ordered_gallery_plots(selected_plots, height_pairs)
    rendered_figures = "\n".join(
        _render_plot_link(plot, title, _format_plot_height_label(plot))
        for plot in ordered_plots
    )
    if rendered_figures == "":
        rendered_figures = '<p>No comparison plots found.</p>'

    plot_count = len(ordered_plots)
    group_count = len({plot.height_pair for plot in ordered_plots})
    summary = f"{plot_count} plots across {group_count} height permutations"
    gaussian_sheet_tight_css = ""
    if comparison_type == "gaussian_filter_comparison":
        gaussian_sheet_tight_css = """
    .figure-sheet {
      padding-left: 8px;
      padding-right: 8px;
    }

    .figure-link {
      width: min(18vw, 220px);
      margin-right: 14px;
      overflow: hidden;
    }

    .figure-link img {
      margin-left: -12%;
      margin-right: -12%;
      max-width: 124%;
    }
"""

    return f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>{escape(title)} Gallery</title>
  <style>
    html,
    body {{
      margin: 0;
      padding: 0;
      background: #ffffff;
      color: #111111;
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    }}

    header {{
      padding: 12px 16px 4px;
    }}

    h1 {{
      margin: 0;
      font-size: 20px;
      font-weight: 600;
      letter-spacing: 0;
    }}

    .summary {{
      margin: 2px 0 0;
      color: #555555;
      font-size: 13px;
      letter-spacing: 0;
    }}

    .figure-sheet {{
      white-space: nowrap;
      padding: 12px 16px 24px;
    }}

    .figure-link {{
      display: inline-block;
      margin-right: 24px;
      vertical-align: top;
      color: inherit;
      text-decoration: none;
      width: min(30vw, 400px);
    }}

    .figure-link:focus-visible {{
      outline: 2px solid #0b5cab;
      outline-offset: 4px;
    }}

    .height-label {{
      display: block;
      margin: 0 0 6px;
      color: #444444;
      font-size: 13px;
      line-height: 1.2;
    }}

    img {{
      display: block;
      width: auto;
      max-width: 100%;
      height: auto;
      image-rendering: auto;
    }}
{gaussian_sheet_tight_css}
  </style>
</head>
<body>
  <header>
    <h1>{escape(title)}</h1>
    <p class="summary">{escape(summary)}</p>
  </header>
  <main class="figure-sheet" aria-label="{escape(title)} height permutations">
{rendered_figures}
  </main>
  <script>
    (() => {{
      const storageKey = `comparison-gallery:${{window.location.pathname}}`;
      const saveScroll = () => {{
        try {{
          window.sessionStorage.setItem(storageKey, JSON.stringify({{
            scrollX: window.scrollX,
            scrollY: window.scrollY,
          }}));
        }} catch (error) {{
        }}
      }};
      const restoreScroll = () => {{
        try {{
          const stored = window.sessionStorage.getItem(storageKey);
          if (stored === null) {{
            return;
          }}
          const state = JSON.parse(stored);
          window.scrollTo(Number(state.scrollX) || 0, Number(state.scrollY) || 0);
        }} catch (error) {{
        }}
      }};

      document.addEventListener("click", (event) => {{
        const link = event.target.closest("a.figure-link");
        if (link === null) {{
          return;
        }}

        event.preventDefault();
        saveScroll();
        window.location.assign(link.href);
      }});

      window.addEventListener("pagehide", saveScroll);
      window.addEventListener("pageshow", restoreScroll);
    }})();
  </script>
</body>
</html>
"""


def write_comparison_viewer(
    figure_dir: str | Path,
    comparison_type: str,
    plots: Iterable[ComparisonPlot],
    height_pairs: Iterable[tuple[float, float]],
) -> dict[str, object]:
    """Write one comparison gallery HTML file."""

    figure_dir = Path(figure_dir).expanduser().resolve()
    figure_dir.mkdir(parents=True, exist_ok=True)
    output_file = figure_dir / COMPARISON_VIEWERS[comparison_type]["output_name"]
    selected_plots = [plot for plot in plots if plot.comparison_type == comparison_type]
    html = render_comparison_view_html(comparison_type, selected_plots, height_pairs)
    output_file.write_text(html, encoding="utf-8")

    view_dir = figure_dir / FULL_RESOLUTION_VIEW_DIRNAME
    view_dir.mkdir(parents=True, exist_ok=True)
    full_res_files: list[Path] = []
    validation_warnings: list[str] = []
    for plot in selected_plots:
        height_label = _format_plot_height_label(plot)
        view_file = view_dir / _full_resolution_view_name(plot)
        full_res_files.append(view_file)
        navigation_targets = build_subplot_navigation_targets(
            plot,
            COMPARISON_VIEWERS[comparison_type]["title"],
            height_label,
            figure_dir=figure_dir,
        )
        validation_warnings.extend(validate_subplot_navigation_targets(plot, navigation_targets, figure_dir))
        view_html = render_full_resolution_view_html(
            plot,
            COMPARISON_VIEWERS[comparison_type]["title"],
            height_label,
            figure_dir=figure_dir,
            navigation_targets=navigation_targets,
        )
        view_file.write_text(view_html, encoding="utf-8")

    warning_log = view_dir / f"{comparison_type}_subplot_link_warnings.log"
    if len(validation_warnings) == 0:
        warning_log.write_text("No subplot link warnings.\n", encoding="utf-8")
    else:
        warning_log.write_text("\n".join(validation_warnings) + "\n", encoding="utf-8")
        for warning in validation_warnings[:25]:
            print(f"Warning: {warning}")
        if len(validation_warnings) > 25:
            print(f"Warning: {len(validation_warnings) - 25} additional subplot link warning(s) written to {warning_log}")

    return {
        "comparison_type": comparison_type,
        "output_file": output_file,
        "plot_count": len(selected_plots),
        "full_resolution_output_files": full_res_files,
        "subplot_link_warning_count": len(validation_warnings),
        "subplot_link_warning_log": warning_log,
    }


def write_default_comparison_viewers(figure_dir: str | Path) -> dict[str, dict[str, object]]:
    """Discover all default comparison plots and write the three gallery viewers."""

    plots = discover_comparison_plots(figure_dir)
    height_pairs = comparison_height_pairs(plots)

    return {
        comparison_type: write_comparison_viewer(
            figure_dir,
            comparison_type,
            plots,
            height_pairs,
        )
        for comparison_type in COMPARISON_VIEWERS
    }


In [ ]:
gallery_outputs = write_default_comparison_viewers(get_figure_dir(config))
for record in gallery_outputs.values():
    print(record['output_file'])
